# EXP-013 multi-source labels + confidence-weighted loss + denser clips

变量: EXP-012 单源 LLM → **steven+lixin+pilk 多源概率混合** + `w=2*|y-0.5|` + `n_clips=4`。
- silence/UNK → NaN；0.5+(p-0.5)*0.75 clip[0.08,0.92]；gold 优先
- 对照: Axial labels005 **0.699** / Axial LLM-single **0.701**
- 存 model.pt，评分走薄推理防超时


In [ ]:
# 提交模式:禁网(enable_internet:false),torch 2.2.0+cu118 + nvidia-cu11 依赖全部离线安装
# 坑1:Kaggle 挂载把文件名中 '+' 去掉(2.2.0cu118=非法 PEP440)→ 复制为合法名
# 坑2:torch 2.2.0+cu118 的 CUDA 库在 nvidia-*-cu11 依赖包中(不在 torch wheel 内),
#     只装 torch 会缺 libcudart(v8/v9 实证)→ 必须显式安装全部 13 个 wheel
# 坑3:/kaggle/working 会被收集为 kernel 输出 → 装完立即删除
import subprocess, sys, os, glob, shutil
wh = sorted(glob.glob('/kaggle/input/**/*.whl', recursive=True))
print('挂载 wheel 共', len(wh), '个')
for w in wh: print('  ', os.path.basename(w))
assert len(wh) >= 13, 'FAIL: 离线 wheel 不足(应 13: torch+torchvision+11 nvidia-cu11)'
fixed = []
for w in wh:
    base = os.path.basename(w)
    dst = os.path.join('/kaggle/working', base.replace('cu118', '+cu118', 1) if 'cu118' in base else base)
    shutil.copy(w, dst)
    fixed.append(dst)
print('安装 wheel:', [os.path.basename(f) for f in fixed])
subprocess.check_call([sys.executable,"-m","pip","uninstall","-y","-q","torch","torchvision","torchaudio"])
# 显式安装全部 wheel(--no-deps --no-index: 无版本解析/无联网)
subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-deps","--no-index"] + fixed)
for f in fixed: os.remove(f)
import numpy, torch, torchvision, sklearn, pandas
print("torch", torch.__version__, "cuda", torch.version.cuda, "| numpy", numpy.__version__)
assert torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# preflight: competition mount + Axial coverage
# Correct layout: test_series/<StudyInstanceUID>/<SeriesInstanceUID>/*.dcm
import os, csv
from collections import defaultdict

base = '/kaggle/input'

def _find(b, depth=0, max_depth=3):
    if not os.path.isdir(b):
        return None
    try:
        entries = sorted(os.listdir(b))
    except OSError:
        return None
    for e in entries:
        p = os.path.join(b, e)
        if not os.path.isdir(p):
            continue
        if e in ('train_series', 'test_series'):
            continue
        if os.path.isfile(os.path.join(p, 'test.csv')) and os.path.isdir(os.path.join(p, 'test_series')):
            return p
        if depth < max_depth - 1:
            r = _find(p, depth + 1)
            if r:
                return r
    return None

root = _find(base)
if root is None:
    raise SystemExit('FAIL: 未找到含 test_series+test.csv 的比赛挂载')
print('root =', root)

def count_dcm(dirpath):
    if not os.path.isdir(dirpath):
        return -1
    return sum(1 for f in os.listdir(dirpath) if f.lower().endswith('.dcm'))

def plane_stats(series_csv, series_dir, label):
    if not os.path.isfile(series_csv):
        print(label, 'series csv missing', series_csv)
        return
    with open(series_csv, newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    if not rows:
        raise SystemExit(f'FAIL: {label} series csv empty')
    plane_key = None
    for k in rows[0].keys():
        if 'plane' in k.lower() or 'anatom' in k.lower():
            plane_key = k
            break
    uid_key = 'StudyInstanceUID'
    ser_key = 'SeriesInstanceUID'
    series_counts = defaultdict(int)
    study_sets = defaultdict(set)
    axial_series_rows = []
    for r in rows:
        pl = r.get(plane_key, '?') if plane_key else '?'
        series_counts[pl] += 1
        if uid_key in r:
            study_sets[pl].add(r[uid_key])
        if pl == 'Axial' and ser_key in r:
            axial_series_rows.append(r)
    print(label, 'plane series counts', dict(series_counts))
    print(label, 'plane study counts', {k: len(v) for k, v in study_sets.items()})
    print(label, 'Axial studies', len(study_sets.get('Axial', set())))
    assert len(study_sets.get('Axial', set())) > 0, f'FAIL: {label} 无 Axial study'

    # Walk both possible layouts
    n_dcm = 0
    hits = 0
    misses = []
    for r in axial_series_rows:
        suid = r[uid_key]
        seid = r[ser_key]
        candidates = [
            os.path.join(series_dir, suid, seid),
            os.path.join(series_dir, seid),
            os.path.join(series_dir, suid, 'series', seid),
        ]
        found = None
        for c in candidates:
            k = count_dcm(c)
            if k and k > 0:
                found = c
                n_dcm += k
                hits += 1
                print(f'  axial OK study={suid[:14]} series={seid[:16]} dcm={k} path={c}')
                break
        if found is None:
            # fallback: list study dir
            sdir = os.path.join(series_dir, suid)
            if os.path.isdir(sdir):
                children = sorted(os.listdir(sdir))[:8]
                print(f'  axial miss study={suid[:14]} series={seid[:16]} study_children={children}')
            else:
                print(f'  axial miss study={suid[:14]} series={seid[:16]} NO_STUDY_DIR {sdir}')
            misses.append(seid)
    print(label, f'axial series hits={hits}/{len(axial_series_rows)} dcm_total={n_dcm}')
    assert hits > 0 and n_dcm > 0, f'FAIL: {label} Axial DICOM 为空 (hits={hits}, dcm={n_dcm})'

    # also sample sagittal for sanity
    sag = [r for r in rows if (r.get(plane_key) if plane_key else '') == 'Sagittal']
    if sag:
        r0 = sag[0]
        for c in [os.path.join(series_dir, r0[uid_key], r0[ser_key]), os.path.join(series_dir, r0[ser_key])]:
            k = count_dcm(c)
            if k and k > 0:
                print(label, 'sagittal sanity', r0[ser_key][:16], 'dcm', k)
                break

plane_stats(os.path.join(root, 'test_series.csv'), os.path.join(root, 'test_series'), 'test')
plane_stats(os.path.join(root, 'train_series.csv'), os.path.join(root, 'train_series'), 'train')
print('preflight axial: OK')


In [ ]:
import base64, os
# embed rsna_v3.py
parts=[]
parts.append("IyAtKi0gY29kaW5nOiB1dGYtOCAtKi0NCiIiIg0KUlNOQSBLbmVlIEFibm9ybWFsaXR5IERldGVjdGlvbiDigJQgdjIo5oyJIEFuYXRvbWljYWxfUGxhbmUg5YiG5bmz6Z2i54us56uL5bu65qihICsg6J6N5ZCIKQ0KPT09PT09PT09PT09PT09PT09PT09PT09PT09")
parts.append("PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0NCuebuOWvuSBiYXNlbGluZShyc25hX2Jhc2VsaW5lLnB5KeeahOaguOW/g+aUueWKqDoNCi0gYmFzZWxpbmUg5oqKIHN0dWR5IOWGheaJgOacieW5s+mdoueahCBzZXJp")
parts.append("ZXMg57KX5pq05ou85oiQ6YCa6YGTO3YyIOaMiSBBbmF0b21pY2FsX1BsYW5lDQogIChBeGlhbCAvIFNhZ2l0dGFsIC8gQ29yb25hbCnlkITorq3kuIDkuKrni6znq4sgUmVzTmV0MzQs5bmz6Z2i5YaF6YOo5YaN5YGaIDIuNUQgY2xpcDsNCi0gdGVzdCDmjqjnkIbm")
parts.append("l7bmr4/kuKrlubPpnaLmqKHlnovni6znq4vpooTmtYss5pyA57uI5oyJIOWQhOW5s+mdoiBPT0Yg5a6P5bmz5Z2HIEFVQyDliqDmnYPono3lkIgNCiAgKOadg+mHjeWPryAtLWZ1c2Ugd2VpZ2h0IOeUqCBBVUMg5Yqg5p2DLOaIliAtLWZ1c2UgbWVhbiDnroDljZXl")
parts.append("ubPlnYcpOw0KLSDmn5Agc3R1ZHkg57y65p+Q5bmz6Z2i5pe2LOiejeWQiOWPquWvueivpSBzdHVkeSDlrp7pmYXlj6/nlKjnmoTlubPpnaLlvZLkuIDljJbmnYPph43jgIINCg0K5YW25L2Z6YC76L6RKOi9r+agh+etvuWhq+WFheOAgUdyb3VwS0ZvbGTjgIF2YWxp")
parts.append("ZCDmjqnnoIHmjZ/lpLHjgIFESUNPTSDpooTlpITnkIYp5LiOIGJhc2VsaW5lIOS4gOiHtOOAgg0KDQrov5DooYznjq/looM6S2FnZ2xlIE5vdGVib29rKEdQVSzmjILovb3mr5TotZvmlbDmja4gKyDnpL7ljLrova/moIfnrb7mlbDmja7pm4Yp44CCDQrnlKjms5U6")
parts.append("DQogICAgIXB5dGhvbiByc25hX3YyLnB5IC0tZXBvY2hzIDMgLS1iYXRjaCAxNiAtLW4tY2xpcHMgMiAtLXBsYW5lcyBzYWdpdHRhbCxjb3JvbmFsDQogICAgIXB5dGhvbiByc25hX3YyLnB5IC0tZXBvY2hzIDMgLS1iYXRjaCAxNiAtLWZ1c2Ugd2VpZ2h0IC0tcHNl")
parts.append("dWRvLWNvbmYgMC4zDQrovpPlh7o6IHN1Ym1pc3Npb24uY3N2KFN0dWR5SW5zdGFuY2VVSUQgKyAxMiDliJfnva7kv6HliIbmlbApDQoiIiINCmltcG9ydCBhcmdwYXJzZQ0KaW1wb3J0IGpzb24NCmltcG9ydCBvcw0KaW1wb3J0IHN5cw0KaW1wb3J0IHRpbWUNCmlt")
parts.append("cG9ydCB3YXJuaW5ncw0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQoNCmltcG9ydCBudW1weSBhcyBucA0KaW1wb3J0IHBhbmRhcyBhcyBwZA0KDQp0cnk6DQogICAgaW1wb3J0IHRvcmNoDQogICAgaW1wb3J0IHRvcmNoLm5uIGFzIG5uDQogICAgZnJvbSB0b3Jj")
parts.append("aC51dGlscy5kYXRhIGltcG9ydCBEYXRhc2V0LCBEYXRhTG9hZGVyDQogICAgZnJvbSB0b3JjaC5vcHRpbSBpbXBvcnQgQWRhbVcNCiAgICBmcm9tIHRvcmNoLm9wdGltLmxyX3NjaGVkdWxlciBpbXBvcnQgQ29zaW5lQW5uZWFsaW5nTFINCiAgICBIQVNfVE9SQ0gg")
parts.append("PSBUcnVlDQpleGNlcHQgSW1wb3J0RXJyb3I6DQogICAgSEFTX1RPUkNIID0gRmFsc2UNCg0KdHJ5Og0KICAgIGltcG9ydCB0b3JjaHZpc2lvbi5tb2RlbHMgYXMgdHZtDQogICAgSEFTX1RWID0gVHJ1ZQ0KZXhjZXB0IEltcG9ydEVycm9yOg0KICAgIEhBU19UViA9")
parts.append("IEZhbHNlDQoNCnRyeToNCiAgICBpbXBvcnQgcHlkaWNvbQ0KICAgIEhBU19QWURJQ09NID0gVHJ1ZQ0KZXhjZXB0IEltcG9ydEVycm9yOg0KICAgIEhBU19QWURJQ09NID0gRmFsc2UNCg0KdHJ5Og0KICAgIGltcG9ydCByc25hX3JlcG9ydF9sYWJlbHMgYXMgUkwg")
parts.append("ICAjIOaKpeWRiuKGkjEyIOeXheeBtuW8seagh+etvijkuI4gcnNuYV92My5weSDlkIznm67lvZUpDQogICAgSEFTX1JMID0gVHJ1ZQ0KZXhjZXB0IEltcG9ydEVycm9yOg0KICAgIEhBU19STCA9IEZhbHNlDQoNCmZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24g")
parts.append("aW1wb3J0IEdyb3VwS0ZvbGQNCmZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCByb2NfYXVjX3Njb3JlDQoNCndhcm5pbmdzLmZpbHRlcndhcm5pbmdzKCJpZ25vcmUiKQ0KDQpMQUJFTFMgPSBbIkFDTCIsICJNQ0wiLCAiTWVkaWFsIE1lbmlzY3VzIiwgIkxhdGVy")
parts.append("YWwgTWVuaXNjdXMiLA0KICAgICAgICAgICJNZWRpYWwgT0EiLCAiTGF0ZXJhbCBPQSIsICJQRiBPQSIsICJFZmZ1c2lvbiIsICJTeW5vdml0aXMiLA0KICAgICAgICAgICJCYWtlcidzIiwgIkNvbnR1c2lvbiIsICJGcmFjdHVyZSJdDQpMQUJFTFNfRU4gPSBbIkFD")
parts.append("TCIsICJNQ0wiLCAiTWVkaWFsTWVuaXNjdXMiLCAiTGF0ZXJhbE1lbmlzY3VzIiwNCiAgICAgICAgICAgICAiTWVkaWFsT0EiLCAiTGF0ZXJhbE9BIiwgIlBGT0EiLCAiRWZmdXNpb24iLCAiU3lub3ZpdGlzIiwNCiAgICAgICAgICAgICAiQmFrZXJzIiwgIkNvbnR1")
parts.append("c2lvbiIsICJGcmFjdHVyZSJdICAjIOWIl+WQjeWuieWFqOWMlijlhpnmj5DkuqTnlKgpDQpQTEFORVMgPSBbIkF4aWFsIiwgIlNhZ2l0dGFsIiwgIkNvcm9uYWwiXQ0KDQpJTUdfU0laRSA9IDIyNA0KQ0xJUCA9IDMgICAgICAgICAgIyDmr48gY2xpcCDliIfniYfm")
parts.append("lbAocHJldi9jZW50ZXIvbmV4dCkNCk1BWF9TRVJJRVMgPSA0ICAgICMg5q+PIHN0dWR5IOavj+S4quW5s+mdouacgOWkmuWPlueahCBzZXJpZXMg5pWwKHYyIOaMieW5s+mdouWPlizljZXlubPpnaIgc2VyaWVzIOabtOWwkSw0IOi2s+WknykNCkZPTERTID0gNQ0K")
parts.append("U0VFRCA9IDQyDQoNCg0KZGVmIHJlc29sdmVfcGF0aHMoKToNCiAgICAiIiLmlbDmja7lrprkvY3pobrluo86MSkgL2thZ2dsZS9pbnB1dCDmjILovb07Mikga2FnZ2xlaHViIOiHquWKqOS4i+i9vTszKSDmnKzlnLAgcnNuYV9rbmVlLyDnm67lvZXjgIIiIiINCiAg")
parts.append("ICAjIDEpIEthZ2dsZSDmjILovb0o5YW85a655aSa5bGC5oyC6L29Oi9rYWdnbGUvaW5wdXQvPOaVsOaNruWQjT4vIOaIliAva2FnZ2xlL2lucHV0LzzliIbnsbs+LzzmlbDmja7lkI0+LykNCiAgICBpZiBvcy5wYXRoLmlzZGlyKCIva2FnZ2xlL2lucHV0Iik6DQog")
parts.append("ICAgICAgIGZvciBzdWIgaW4gc29ydGVkKG9zLmxpc3RkaXIoIi9rYWdnbGUvaW5wdXQiKSk6DQogICAgICAgICAgICBkID0gb3MucGF0aC5qb2luKCIva2FnZ2xlL2lucHV0Iiwgc3ViKQ0KICAgICAgICAgICAgZm9yIGNhbmQgaW4gKGQsIG9zLnBhdGguam9pbihk")
parts.append("LCBzdWIpKTogICMg5aSE55CGIEthZ2dsZSDlj4zlsYLmjILovb0NCiAgICAgICAgICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhvcy5wYXRoLmpvaW4oY2FuZCwgInRyYWluLmNzdiIpKToNCiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIGNhbmQsICIva2FnZ2xl")
parts.append("L3dvcmtpbmciDQogICAgICAgICAgICBpZiBvcy5wYXRoLmlzZGlyKGQpOiAgIyDlho3lvoDkuIvkuIDlsYLmib4o5aaCIC9rYWdnbGUvaW5wdXQvY29tcGV0aXRpb25zLzxuYW1lPi8pDQogICAgICAgICAgICAgICAgZm9yIHN1YjIgaW4gc29ydGVkKG9zLmxpc3Rk")
parts.append("aXIoZCkpOg0KICAgICAgICAgICAgICAgICAgICBjYW5kID0gb3MucGF0aC5qb2luKGQsIHN1YjIpDQogICAgICAgICAgICAgICAgICAgIGlmIG9zLnBhdGguZXhpc3RzKG9zLnBhdGguam9pbihjYW5kLCAidHJhaW4uY3N2IikpOg0KICAgICAgICAgICAgICAgICAg")
parts.append("ICAgICAgcmV0dXJuIGNhbmQsICIva2FnZ2xlL3dvcmtpbmciDQogICAgICAgIHByaW50KCJb6K+K5patXSAva2FnZ2xlL2lucHV0IOWtmOWcqCzkvYbmsqHmib7liLDlkKsgdHJhaW4uY3N2IOeahOebruW9leOAguWunumZheaMgui9veWGheWuueWmguS4izoiLCBm")
parts.append("bHVzaD1UcnVlKQ0KICAgICAgICBmb3Igc3ViIGluIHNvcnRlZChvcy5saXN0ZGlyKCIva2FnZ2xlL2lucHV0IikpOg0KICAgICAgICAgICAgcCA9IG9zLnBhdGguam9pbigiL2thZ2dsZS9pbnB1dCIsIHN1YikNCiAgICAgICAgICAgIHRyeToNCiAgICAgICAgICAg")
parts.append("ICAgICBpbm5lciA9IG9zLmxpc3RkaXIocCkNCiAgICAgICAgICAgIGV4Y2VwdCBPU0Vycm9yOg0KICAgICAgICAgICAgICAgIGlubmVyID0gWyIo5peg5rOV6K+75Y+WKSJdDQogICAgICAgICAgICBwcmludChmIiAgL2thZ2dsZS9pbnB1dC97c3VifS8gIC0+ICB7")
parts.append("aW5uZXJbOjhdfSIsIGZsdXNoPVRydWUpDQogICAgICAgIHByaW50KCJb6K+K5patXSDlpoLmnpzkvaDmjILnmoTmmK8gJ+S9nOiAheaVsOaNruWMhS/liKvkurrnmoQga2VybmVsIOi+k+WHuicsIOmHjOmdouayoeaciSB0cmFpbi5jc3Yg5ZKMIHRyYWluX3Nlcmll")
parts.append("cy8sIiwNCiAgICAgICAgICAgICAgIuW/hemhu+aMguavlOi1m+aVsOaNruacrOS9kyByc25hLWtuZWUtYWJub3JtYWxpdHktZGV0ZWN0aW9uIChBZGQgSW5wdXQgLT4gQ29tcGV0aXRpb25zIOagh+etvumhtSkiLCBmbHVzaD1UcnVlKQ0KICAgICMgMikga2FnZ2xl")
parts.append("aHViIOWFnOW6lTrmnKrmjILovb3ml7boh6rliqjkuIvovb3mr5TotZvmlbDmja4oS2FnZ2xlIE5vdGVib29rIOWGhee9ruWHreaNrikNCiAgICB0cnk6DQogICAgICAgIGltcG9ydCBrYWdnbGVodWINCiAgICAgICAgZCA9IGthZ2dsZWh1Yi5jb21wZXRpdGlvbl9k")
parts.append("b3dubG9hZCgicnNuYS1rbmVlLWFibm9ybWFsaXR5LWRldGVjdGlvbiIpDQogICAgICAgIGlmIG9zLnBhdGguZXhpc3RzKG9zLnBhdGguam9pbihkLCAidHJhaW4uY3N2IikpOg0KICAgICAgICAgICAgcHJpbnQoZiJb6Lev5b6EXSDmnKrmjILovb0s5bey55SoIGth")
parts.append("Z2dsZWh1YiDojrflj5bmlbDmja46IHtkfSIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICByZXR1cm4gZCwgIi9rYWdnbGUvd29ya2luZyINCiAgICBleGNlcHQgSW1wb3J0RXJyb3I6DQogICAgICAgIHBhc3MgICMg5pys5Zyw5pyq6KOFIGthZ2dsZWh1Yg0KICAg")
parts.append("IGV4Y2VwdCBFeGNlcHRpb24gYXMgZToNCiAgICAgICAgcHJpbnQoZiJb5o+Q56S6XSBrYWdnbGVodWIg5LiL6L295pWw5o2u5aSx6LSlOiB7ZX0iLCBmbHVzaD1UcnVlKQ0KICAgICMgMykg5pys5Zyw5Zue6YCAKOiEmuacrOebruW9lSAvIGN3ZCAvIGN3ZCDkuIvn")
parts.append("moQgcnNuYV9rbmVlIOWtkOebruW9lemDveaJvikNCiAgICBjYW5kcyA9IFtdDQogICAgdHJ5Og0KICAgICAgICBjYW5kcy5hcHBlbmQob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKQ0KICAgIGV4Y2VwdCBOYW1lRXJyb3I6DQogICAg")
parts.append("ICAgIHBhc3MNCiAgICBjYW5kcy5hcHBlbmQob3MuZ2V0Y3dkKCkpDQogICAgZm9yIGJhc2UgaW4gbGlzdChjYW5kcyk6DQogICAgICAgIGNhbmRzLmFwcGVuZChvcy5wYXRoLmpvaW4oYmFzZSwgInJzbmFfa25lZSIpKQ0KICAgIHNlZW4gPSBzZXQoKQ0KICAgIGZv")
parts.append("ciBiYXNlIGluIGNhbmRzOg0KICAgICAgICBpZiBiYXNlIGluIHNlZW46DQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICBzZWVuLmFkZChiYXNlKQ0KICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhvcy5wYXRoLmpvaW4oYmFzZSwgInRyYWluLmNzdiIpKToN")
parts.append("CiAgICAgICAgICAgIHJldHVybiBiYXNlLCBiYXNlDQogICAgc3lzLmV4aXQoIuaJvuS4jeWIsOaVsOaNrjror7flhYjlnKjmr5TotZvpobXliqDlhaXmr5TotZvlubblkIzmhI/mlbDmja7op4TliJks5YaNIEFkZCBJbnB1dCDmjILovb0gIg0KICAgICAgICAgICAg")
parts.append("ICJyc25hLWtuZWUtYWJub3JtYWxpdHktZGV0ZWN0aW9uO+acrOWcsOivt+WcqCByc25hX2tuZWUvIOS4i+i/kOihjOOAgiIpDQoNCg0KX1BTRVVET19GSUxFUyA9ICgidHJhaW5fZm9sZHNfd2l0aF9wc2V1ZG8uY3N2IiwgInRyYWluX3dpdGhfcHNldWRvLmNzdiIs")
parts.append("DQogICAgICAgICAgICAgICAgICJ0cmFpbl93aXRoX3NvZnRfbGFiZWxzLmNzdiIsICJ0cmFpbl9wc2V1ZG8uY3N2IikNCl9TS0lQX0RJUlMgPSB7InRyYWluX3NlcmllcyIsICJ0ZXN0X3NlcmllcyJ9ICAjIERJQ09NIOWkp+ebruW9lSznu53kuI3ljrvpgY3ljoYN")
parts.append("Cl9XRUlHSFRfRVhUID0gKCIucHRoIiwgIi5wdCIsICIuY2twdCIsICIucHRoLnRhciIpICAjIOemu+e6v+mihOiuree7g+adg+mHjeaWh+S7tuaJqeWxleWQjQ0KDQojIDEyIOagh+etvuWkmuivreiogOWFs+mUruivjeivjeWFuCjoi7Ev6KW/L+iNty/lvrcv5rOV")
parts.append("KeKAlOKAlOmbtuaMgui9vei9r+agh+etvuWFnOW6lQ0KIyDlkb3kuK0g4oaSIOmrmOe9ruS/oSzmnKrlkb3kuK0g4oaSIOS9jue9ruS/oSjova/moIfnrb4s5LiN57uZ56GsIDAvMSzpgb/lhY3plJnor43or6/lr7wpDQpfUkVQT1JUX0tFWVdPUkRTID0gew0KICAg")
parts.append("ICJBQ0wiOiBbImFjbCIsICJhbnRlcmlvciBjcnVjaWF0ZSIsICJjcnVjaWFkbyBhbnRlcmlvciIsICJ2b29yc3RlIGtydWlzYmFuZCIsDQogICAgICAgICAgICAidm9yZGVyZW4ga3JldXpiYW5kIiwgImxjYSIsICJsaWdhbWVudCBjcm9pc8OpIGFudMOpcmlldXIi")
parts.append("XSwNCiAgICAiTUNMIjogWyJtY2wiLCAibWVkaWFsIGNvbGxhdGVyYWwiLCAiY29sYXRlcmFsIG1lZGlhbCIsICJtZWRpYWxlIGNvbGxhdGVyYWxlIiwNCiAgICAgICAgICAgICJpbm5lcmVzIHNlaXRlbmJhbmQiLCAibGlnYW1lbnQgY29sbGF0w6lyYWwgbcOpZGlh")
parts.append("bCJdLA0KICAgICJNZWRpYWwgTWVuaXNjdXMiOiBbIm1lZGlhbCBtZW5pc2N1cyIsICJtZW5pc2NvIGludGVybm8iLCAibWVuaXNjdXMgbWVkaWFsaXMiLA0KICAgICAgICAgICAgICAgICAgICAgICAgIm1lbmlza3VzIG1lZGlhbCIsICJtZW5pc3F1ZSBtw6lkaWFs")
parts.append("IiwgImlubmVyIG1lbmlzY3VzIiwgIm1lbmlzY28gbWVkaWFsIl0sDQogICAgIkxhdGVyYWwgTWVuaXNjdXMiOiBbImxhdGVyYWwgbWVuaXNjdXMiLCAibWVuaXNjbyBleHRlcm5vIiwgIm1lbmlzY3VzIGxhdGVyYWxpcyIsDQogICAgICAgICAgICAgICAgICAgICAg")
parts.append("ICAgIm1lbmlza3VzIGxhdGVyYWwiLCAibWVuaXNxdWUgbGF0w6lyYWwiLCAib3V0ZXIgbWVuaXNjdXMiXSwNCiAgICAiTWVkaWFsIE9BIjogWyJtZWRpYWwgb3N0ZW9hcnRocml0aXMiLCAiYXJ0cm9zaXMgZmVtb3JvdGliaWFsIG1lZGlhbCIsDQogICAgICAgICAg")
parts.append("ICAgICAgICAibWVkaWFsZSBnb25hcnRyb3NlIiwgIm1lZGlhbGUgYXJ0aHJvc2UiLCAib3N0ZW9hcnRyaXRpcyBtZWRpYWwiLA0KICAgICAgICAgICAgICAgICAgIm1lZGlhbCBjb21wYXJ0bWVudCBuYXJyb3dpbmciLCAibWVkaWFsIGpvaW50IHNwYWNlIG5hcnJv")
parts.append("d2luZyJdLA0KICAgICJMYXRlcmFsIE9BIjogWyJsYXRlcmFsIG9zdGVvYXJ0aHJpdGlzIiwgImFydHJvc2lzIGZlbW9yb3RpYmlhbCBsYXRlcmFsIiwNCiAgICAgICAgICAgICAgICAgICAibGF0ZXJhbGUgZ29uYXJ0cm9zZSIsICJsYXRlcmFsZSBhcnRocm9zZSIs")
parts.append("ICJsYXRlcmFsIGpvaW50IHNwYWNlIG5hcnJvd2luZyJdLA0KICAgICJQRiBPQSI6IFsicGF0ZWxsb2ZlbW9yYWwgb3N0ZW9hcnRocml0aXMiLCAiYXJ0cm9zaXMgcGF0ZWxvZmVtb3JhbCIsDQogICAgICAgICAgICAgICJwYXRlbGxvZmVtb3JhbGUgYXJ0cm9zZSIs")
parts.append("ICJwYXRlbGxvZmVtb3JhbCBhcnRocm9zZSIsICJwZiBvYSIsDQogICAgICAgICAgICAgICJwYXRlbGxvZmVtb3JhbCBuYXJyb3dpbmciLCAicGF0ZWxsb2ZlbW9yYWwgam9pbnQgc3BhY2UiXSwNCiAgICAiRWZmdXNpb24iOiBbImVmZnVzaW9uIiwgImRlcnJhbWUi")
parts.append("LCAidm9jaHQiLCAiZXJndXNzIiwgIsOpcGFuY2hlbWVudCIsICJqb2ludCBlZmZ1c2lvbiJdLA0KICAgICJTeW5vdml0aXMiOiBbInN5bm92aXRpcyIsICJzaW5vdml0aXMiLCAic3lub3ZpYWxlIiwgInN5bm92aWFsIHRoaWNrZW5pbmciLCAic3lub3ZpdGlzIl0s")
parts.append("DQogICAgIkJha2VyJ3MiOiBbImJha2VyIiwgInBvcGxpdGVhbCBjeXN0IiwgInF1aXN0ZSBkZSBiYWtlciIsICJiYWtlciBjeXN0Il0sDQogICAgIkNvbnR1c2lvbiI6IFsiY29udHVzaW9uIiwgImJvbmUgYnJ1aXNlIiwgImNvbnR1c2nDs24iLCAia25ldXppbmci")
parts.append("LCAia29udHVzaW9uIiwNCiAgICAgICAgICAgICAgICAgICJtYXJyb3cgZWRlbWEiLCAiw7NzZW8iLCAiYm9uZSBicnVpc2luZyJdLA0KICAgICJGcmFjdHVyZSI6IFsiZnJhY3R1cmUiLCAiZnJhY3R1cmEiLCAiZnJhY3R1dXIiLCAiZnJha3R1ciIsICJmcmFjdHVy")
parts.append("ZSBsaW5lIiwNCiAgICAgICAgICAgICAgICAgImZyYWN0dXJlIGxpbmUiLCAic3ViY2hvbmRyYWwgZnJhY3R1cmUiXSwNCn0NCg0KDQpkZWYgX3BzZXVkb19mcm9tX3JlcG9ydHModHJhaW5fY3N2LCBjb25mX2hpdD0wLjksIGNvbmZfbWlzcz0wLjEpOg0KICAgICIi")
parts.append("IuS7juaUvuWwhOaKpeWRiuaWh+acrOeUqOWkmuivreiogOWFs+mUruivjeeUn+aIkCAxMiDmoIfnrb7ova/moIfnrb4o6Zu25oyC6L295YWc5bqVKeOAgg0KDQogICAg6L+U5ZueIERhdGFGcmFtZShTdHVkeUluc3RhbmNlVUlEICsgcHNldWRvXyog5YiXKeOAguWF")
parts.append("s+mUruivjeWRveS4reKGkmNvbmZfaGl0LA0KICAgIOacquWRveS4reKGkmNvbmZfbWlzcyjkuI3nu5nnoawgMC8xLOmZjeS9jumUmeivjeivr+WvvCnjgILlrpjmlrnlt7LmnInmoIfms6jnmoQgc3R1ZHkNCiAgICDku43ku6XlrpjmlrnkuLrlh4YobG9hZF9sYWJl")
parts.append("bHMg55qE5aGr5YWF6YC76L6R5L+d6K+B5a6Y5pa55LyY5YWIKeOAgg0KICAgICIiIg0KICAgIGlmICJSZXBvcnQiIG5vdCBpbiB0cmFpbl9jc3YuY29sdW1uczoNCiAgICAgICAgcmV0dXJuIE5vbmUNCiAgICByZXAgPSB0cmFpbl9jc3ZbIlJlcG9ydCJdLmZpbGxu")
parts.append("YSgiIikNCiAgICByb3dzID0ge30NCiAgICBmb3IgbGFiLCBrd3MgaW4gX1JFUE9SVF9LRVlXT1JEUy5pdGVtcygpOg0KICAgICAgICBsb3cgPSBbay5sb3dlcigpIGZvciBrIGluIGt3c10NCiAgICAgICAgcm93c1tmInBzZXVkb197bGFifSJdID0gcmVwLmFwcGx5")
parts.append("KA0KICAgICAgICAgICAgbGFtYmRhIHQ6IGNvbmZfaGl0IGlmIGFueShrIGluIHN0cih0KS5sb3dlcigpIGZvciBrIGluIGxvdykgZWxzZSBjb25mX21pc3MpDQogICAgZGYgPSBwZC5EYXRhRnJhbWUoeyJTdHVkeUluc3RhbmNlVUlEIjogdHJhaW5fY3N2WyJTdHVk")
parts.append("eUluc3RhbmNlVUlEIl0sICoqcm93c30pDQogICAgcmV0dXJuIGRmDQoNCg0KZGVmIF9wcmludF9tb3VudF90cmVlKGJhc2U9Ii9rYWdnbGUvaW5wdXQiLCB0YWc9IuaVsOaNriIsIG1heF9maWxlcz02KToNCiAgICAiIiLmiZPljbDmjILovb3nm67lvZXmoJEo5pyA")
parts.append("5aSa5Lik5bGCLOavj+ebruW9leWJjeWHoOS4quaWh+S7tiks5biu55So5oi356Gu6K6k5oyC6L295YaF5a6544CCIiIiDQogICAgaWYgbm90IG9zLnBhdGguaXNkaXIoYmFzZSk6DQogICAgICAgIHByaW50KGYiW+iviuaWrS17dGFnfV0g55uu5b2V5LiN5a2Y5Zyo")
parts.append("OiB7YmFzZX0iLCBmbHVzaD1UcnVlKQ0KICAgICAgICByZXR1cm4NCiAgICBwcmludChmIlvor4rmlq0te3RhZ31dIC9rYWdnbGUvaW5wdXQg5a6e6ZmF5oyC6L295YaF5a65OiIsIGZsdXNoPVRydWUpDQogICAgZm9yIHN1YiBpbiBzb3J0ZWQob3MubGlzdGRpcihi")
parts.append("YXNlKSk6DQogICAgICAgIHAgPSBvcy5wYXRoLmpvaW4oYmFzZSwgc3ViKQ0KICAgICAgICB0cnk6DQogICAgICAgICAgICBpZiBvcy5wYXRoLmlzZGlyKHApOg0KICAgICAgICAgICAgICAgIGlubmVyID0gc29ydGVkKG9zLmxpc3RkaXIocCkpWzptYXhfZmlsZXNd")
parts.append("DQogICAgICAgICAgICAgICAgcHJpbnQoZiIgIHtzdWJ9LyAtPiB7aW5uZXJ9IiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgICAgIGVsc2U6DQogICAgICAgICAgICAgICAgcHJpbnQoZiIgIHtzdWJ9ICjmlofku7YpIiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgZXhjZXB0")
parts.append("IE9TRXJyb3I6DQogICAgICAgICAgICBwcmludChmIiAge3N1Yn0gKOaXoOazleivu+WPlikiLCBmbHVzaD1UcnVlKQ0KDQoNCmRlZiBfcHJpbnRfZGNtX3RyZWUocm9vdCwgdGFnPSJESUNPTSIsIG1heF9zaG93PTUpOg0KICAgICIiIuaJk+WNsCB0cmFpbl9zZXJp")
parts.append("ZXMg55uu5b2V5YmN5Yeg5bGC55yf5a6e57uT5p6EICsg5oq95qC3IC5kY20g6K6h5pWwLOehruiupCBESUNPTSDluIPlsYDjgIINCg0KICAgIFJTTkEg5bi46KeB5biD5bGAOnRyYWluX3Nlcmllcy88U3R1ZHlJbnN0YW5jZVVJRD4vPFNlcmllc0luc3RhbmNlVUlE")
parts.append("Pi8qLmRjbSjkuKTlsYIpOw0KICAgIOS5n+WFvOWuueS4gOWxgig8U2VyaWVzSW5zdGFuY2VVSUQ+LyouZGNtKeaIluabtOa3seWxgizov5nph4zmiZPljbDlrp7pmYXnnIvliLDnmoTjgIINCiAgICAiIiINCiAgICBkY20gPSBvcy5wYXRoLmpvaW4ocm9vdCwgInRy")
parts.append("YWluX3NlcmllcyIpDQogICAgaWYgbm90IG9zLnBhdGguaXNkaXIoZGNtKToNCiAgICAgICAgcHJpbnQoZiJb6K+K5patLXt0YWd9XSB7ZGNtfSDkuI3lrZjlnKghIiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgcmV0dXJuDQogICAgdHJ5Og0KICAgICAgICBzdWJzX2Fs")
parts.append("bCA9IHNvcnRlZChvcy5saXN0ZGlyKGRjbSkpDQogICAgZXhjZXB0IE9TRXJyb3I6DQogICAgICAgIHN1YnNfYWxsID0gW10NCiAgICBzdWJzID0gc3Vic19hbGxbOm1heF9zaG93XQ0KICAgIHByaW50KGYiW+iviuaWrS17dGFnfV0ge2RjbX0iLCBmbHVzaD1UcnVl")
parts.append("KQ0KICAgIHByaW50KGYiW+iviuaWrS17dGFnfV0g5a2Q55uu5b2VL+aWh+S7tuaVsDoge2xlbihzdWJzX2FsbCl9LOagt+acrDoiLCBmbHVzaD1UcnVlKQ0KICAgIGZvciBzIGluIHN1YnM6DQogICAgICAgIHNwID0gb3MucGF0aC5qb2luKGRjbSwgcykNCiAgICAg")
parts.append("ICAgaWYgb3MucGF0aC5pc2RpcihzcCk6DQogICAgICAgICAgICB0cnk6DQogICAgICAgICAgICAgICAgaW5uZXIgPSBzb3J0ZWQob3MubGlzdGRpcihzcCkpWzptYXhfc2hvd10NCiAgICAgICAgICAgIGV4Y2VwdCBPU0Vycm9yOg0KICAgICAgICAgICAgICAgIGlu")
parts.append("bmVyID0gW10NCiAgICAgICAgICAgIHByaW50KGYiICB7c30vIC0+IHtpbm5lcn0iLCBmbHVzaD1UcnVlKQ0KICAgICAgICAgICAgZm9yIGl0IGluIGlubmVyWzoyXToNCiAgICAgICAgICAgICAgICBpdHAgPSBvcy5wYXRoLmpvaW4oc3AsIGl0KQ0KICAgICAgICAg")
parts.append("ICAgICAgIGlmIG9zLnBhdGguaXNkaXIoaXRwKToNCiAgICAgICAgICAgICAgICAgICAgdHJ5Og0KICAgICAgICAgICAgICAgICAgICAgICAgZmlsZXMgPSBzb3J0ZWQob3MubGlzdGRpcihpdHApKVs6M10NCiAgICAgICAgICAgICAgICAgICAgZXhjZXB0IE9TRXJy")
parts.append("b3I6DQogICAgICAgICAgICAgICAgICAgICAgICBmaWxlcyA9IFtdDQogICAgICAgICAgICAgICAgICAgIHByaW50KGYiICAgICAge2l0fS8gLT4ge2ZpbGVzfSIsIGZsdXNoPVRydWUpDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICBwcmludChmIiAge3N9ICjm")
parts.append("lofku7YpIiwgZmx1c2g9VHJ1ZSkNCiAgICAjIOaKveagtyAuZGNtIOiuoeaVsDrliY0gMyBzdHVkeSDDlyDliY0gMyBzZXJpZXMNCiAgICBuX2RjbSA9IDANCiAgICB0cnk6DQogICAgICAgIGZvciBzIGluIHN1YnNfYWxsWzozXToNCiAgICAgICAgICAgIHNwID0g")
parts.append("b3MucGF0aC5qb2luKGRjbSwgcykNCiAgICAgICAgICAgIGlmIG5vdCBvcy5wYXRoLmlzZGlyKHNwKToNCiAgICAgICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICAgICAgZm9yIGl0IGluIHNvcnRlZChvcy5saXN0ZGlyKHNwKSlbOjNdOg0KICAgICAgICAgICAg")
parts.append("ICAgIGl0cCA9IG9zLnBhdGguam9pbihzcCwgaXQpDQogICAgICAgICAgICAgICAgaWYgb3MucGF0aC5pc2RpcihpdHApOg0KICAgICAgICAgICAgICAgICAgICBuX2RjbSArPSBzdW0oMSBmb3IgZiBpbiBvcy5saXN0ZGlyKGl0cCkgaWYgZi5sb3dlcigpLmVuZHN3")
parts.append("aXRoKCIuZGNtIikpDQogICAgZXhjZXB0IE9TRXJyb3I6DQogICAgICAgIHBhc3MNCiAgICBwcmludChmIlvor4rmlq0te3RhZ31dIOaKveagtyAuZGNtIOaWh+S7tuaVsDoge25fZGNtfSjliY0gMyBzdHVkeSDDlyDliY0gMyBzZXJpZXMpIiwgZmx1c2g9VHJ1ZSkN")
parts.append("Cg0KDQpkZWYgX3NjYW5fb2ZmbGluZV93ZWlnaHRzKGJhc2U9Ii9rYWdnbGUvaW5wdXQiLCBkZXB0aD0wLCBtYXhfZGVwdGg9NCk6DQogICAgIiIi6YCS5b2S5omr5o+P5oyC6L2955uu5b2V5LiL55qE6aKE6K6t57uD5p2D6YeN5paH5Lu2KC5wdGgvLnB0IOetiSks")
parts.append("6Lez6L+HIERJQ09NIOWkp+ebruW9leOAgg0KDQogICAg6L+U5ZueIFso57ud5a+56Lev5b6ELCDnm7jlr7kgL2thZ2dsZS9pbnB1dCDnmoTmj4/ov7ApXTvmib7kuI3liLDov5Tlm57nqbrliJfooajjgIINCiAgICDnpL7ljLrmiZPljIXnmoTnprvnur/mnYPph43m")
parts.append("lbDmja7pm4Yo5aaCIG1hdHRlb25hY2NhcmF0by9yZXNuZXQzNC1pbWFnZW5ldC1wdGjjgIENCiAgICBtYXJjc2hhZGUvcmVzbmV0LXByZXRyYWluZWQtd2VpZ2h0cy1pbWFnZW5ldOOAgXB5dG9yY2gvcmVzbmV0MzQp5oyC6L295ZCO5Y2z5Y+v6KKr5omr5YiwLA0K")
parts.append("ICAgIOaXoOmcgOiBlOe9keOAgg0KICAgICIiIg0KICAgIGhpdHMgPSBbXQ0KICAgIGlmIGRlcHRoID4gbWF4X2RlcHRoIG9yIG5vdCBvcy5wYXRoLmlzZGlyKGJhc2UpOg0KICAgICAgICByZXR1cm4gaGl0cw0KICAgIHRyeToNCiAgICAgICAgZW50cmllcyA9IHNv")
parts.append("cnRlZChvcy5saXN0ZGlyKGJhc2UpKQ0KICAgIGV4Y2VwdCBPU0Vycm9yOg0KICAgICAgICByZXR1cm4gaGl0cw0KICAgIGZvciBmbiBpbiBlbnRyaWVzOg0KICAgICAgICBwID0gb3MucGF0aC5qb2luKGJhc2UsIGZuKQ0KICAgICAgICBpZiBvcy5wYXRoLmlzZmls")
parts.append("ZShwKSBhbmQgZm4ubG93ZXIoKS5lbmRzd2l0aChfV0VJR0hUX0VYVCk6DQogICAgICAgICAgICBoaXRzLmFwcGVuZCgocCwgb3MucGF0aC5yZWxwYXRoKHAsICIva2FnZ2xlL2lucHV0IikpKQ0KICAgICAgICBlbGlmIG9zLnBhdGguaXNkaXIocCkgYW5kIGZuIG5v")
parts.append("dCBpbiBfU0tJUF9ESVJTOg0KICAgICAgICAgICAgaGl0cy5leHRlbmQoX3NjYW5fb2ZmbGluZV93ZWlnaHRzKHAsIGRlcHRoICsgMSwgbWF4X2RlcHRoKSkNCiAgICByZXR1cm4gaGl0cw0KDQoNCmRlZiBfbG9hZF9zdGF0ZV9kaWN0X2Zyb21fZmlsZShwYXRoKToN")
parts.append("CiAgICAiIiLku47mnYPph43mlofku7bor7sgc3RhdGVfZGljdCjlhbzlrrk655u05o6lIHN0YXRlX2RpY3QgLyDljIXlnKggY2hlY2twb2ludCDph4wgLyDlkITnp43liY3nvIAp44CCDQoNCiAgICDor7vlj5blpLHotKXmiJbmoLzlvI/kuI3lr7nov5Tlm54gTm9u")
parts.append("ZeOAgg0KICAgICIiIg0KICAgIHRyeToNCiAgICAgICAgb2JqID0gdG9yY2gubG9hZChwYXRoLCBtYXBfbG9jYXRpb249ImNwdSIsIHdlaWdodHNfb25seT1UcnVlKQ0KICAgIGV4Y2VwdCBUeXBlRXJyb3I6ICAjIOaXp+eJiCB0b3JjaCDml6Agd2VpZ2h0c19vbmx5")
parts.append("IOWPguaVsA0KICAgICAgICBvYmogPSB0b3JjaC5sb2FkKHBhdGgsIG1hcF9sb2NhdGlvbj0iY3B1IikNCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6DQogICAgICAgIHByaW50KGYiW+adg+mHjV0ge3BhdGh9IOivu+WPluWksei0pToge2V9IiwgZmx1c2g9VHJ1")
parts.append("ZSkNCiAgICAgICAgcmV0dXJuIE5vbmUNCiAgICBpZiBpc2luc3RhbmNlKG9iaiwgZGljdCkgYW5kICJzdGF0ZV9kaWN0IiBpbiBvYmo6DQogICAgICAgIG9iaiA9IG9ialsic3RhdGVfZGljdCJdDQogICAgZWxpZiBpc2luc3RhbmNlKG9iaiwgZGljdCkgYW5kICJt")
parts.append("b2RlbF9zdGF0ZV9kaWN0IiBpbiBvYmo6DQogICAgICAgIG9iaiA9IG9ialsibW9kZWxfc3RhdGVfZGljdCJdDQogICAgZWxpZiBpc2luc3RhbmNlKG9iaiwgZGljdCkgYW5kICJtb2RlbCIgaW4gb2JqIGFuZCBpc2luc3RhbmNlKG9ialsibW9kZWwiXSwgZGljdCk6")
parts.append("DQogICAgICAgIG9iaiA9IG9ialsibW9kZWwiXQ0KICAgIGlmIG5vdCBpc2luc3RhbmNlKG9iaiwgZGljdCk6DQogICAgICAgIHJldHVybiBOb25lDQogICAgIyDliaXluLjop4HliY3nvIA6bW9kdWxlLiAvIG1vZGVsLiAvIGJhY2tib25lLijoi6XmlbTkvZPluKbl")
parts.append("iY3nvIAs57uf5LiA6L+Y5Y6fKQ0KICAgIG91dCA9IHt9DQogICAgZm9yIGssIHYgaW4gb2JqLml0ZW1zKCk6DQogICAgICAgIGZvciBwcmUgaW4gKCJtb2R1bGUuIiwgIm1vZGVsLiIsICJiYWNrYm9uZS4iKToNCiAgICAgICAgICAgIGlmIGsuc3RhcnRzd2l0aChw")
parts.append("cmUpOg0KICAgICAgICAgICAgICAgIGsgPSBrW2xlbihwcmUpOl0NCiAgICAgICAgICAgICAgICBicmVhaw0KICAgICAgICBvdXRba10gPSB2DQogICAgcmV0dXJuIG91dA0KDQoNCmRlZiBfdHJ5X2xvYWRfb2ZmbGluZShuZXQsIHBhdGhzKToNCiAgICAiIiLmjInl")
parts.append("uo/lsJ3or5XmiorlgJnpgInnprvnur/mnYPph43liqDovb3ov5sgbmV0KHN0YXRlX2RpY3Qg57uT5p6E6aG75LiOIFJlc05ldDM0IOWMuemFjSnjgIINCg0KICAgIOmAgumFjeinhOWImSjpnaLlkJHnpL7ljLogUlNOQSDmnYPph40s5aaCIGJhcnVuMjEwNCDnmoQg")
parts.append("YmVzdF9yZXNuZXQzNF9mb2xkMC5wdGgpOg0KICAgIC0gY29udjEud2VpZ2h0IOmAmumBk+aVsOS4jeWQjOaXtjpyZXNoYXBlK21lYW4g6YCC6YWN5YiwIG5ldCDnmoQgY29udjEg6YCa6YGT5pWwKOWmgiAyNOKGkjMsDQogICAgICBiYXJ1bjIxMDQg5pivIDggY2xp")
parts.append("cCDDlyAzIOmAmumBk+eahCAyLjVEIOaooeWeiyk7DQogICAgLSBmYy4qIC8gaGVhZC4qIC8gY2xhc3NpZmllci4qIOS4gOW+i+WJpeemuyjmiJHku6znmoTmqKHlnovkvJrmm7/mjaLmiJAgMTIg5aS0KTsNCiAgICDov5Tlm54gKOaYr+WQpuaIkOWKnywg5ZG95Lit")
parts.append("5o+P6L+wKeOAgg0KICAgICIiIg0KICAgIGZvciBwLCBkZXNjIGluIHBhdGhzOg0KICAgICAgICBzZCA9IF9sb2FkX3N0YXRlX2RpY3RfZnJvbV9maWxlKHApDQogICAgICAgIGlmIHNkIGlzIE5vbmU6DQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICAjIDEp")
parts.append("IOmAgumFjSBjb252MTrpgJrpgZPmlbDkuI3lkIzliJkgcmVzaGFwZSttZWFuKOWmgiAyNCDpgJrpgZMg4oaSIDMg6YCa6YGTKQ0KICAgICAgICBpZiAiY29udjEud2VpZ2h0IiBpbiBzZDoNCiAgICAgICAgICAgIHcgPSBzZFsiY29udjEud2VpZ2h0Il0NCiAgICAg")
parts.append("ICAgICAgIHRhcmdldCA9IG5ldC5jb252MS53ZWlnaHQuc2hhcGVbMV0NCiAgICAgICAgICAgIGlmIHcuc2hhcGVbMV0gIT0gdGFyZ2V0IGFuZCB3LnNoYXBlWzFdICUgdGFyZ2V0ID09IDA6DQogICAgICAgICAgICAgICAgQiwgQywgSCwgVyA9IHcuc2hhcGUNCiAg")
parts.append("ICAgICAgICAgICAgICB3MiA9IHcucmVzaGFwZShCLCB0YXJnZXQsIEMgLy8gdGFyZ2V0LCBILCBXKS5tZWFuKGRpbT0yKQ0KICAgICAgICAgICAgICAgIHNkWyJjb252MS53ZWlnaHQiXSA9IHcyDQogICAgICAgICAgICAgICAgcHJpbnQoZiJb5p2D6YeNXSB7ZGVz")
parts.append("Y30gY29udjEg6YCa6YGTIHtDfeKGknt0YXJnZXR9IOW5s+Wdh+mAgumFjSIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICBlbGlmIHcuc2hhcGVbMV0gIT0gdGFyZ2V0Og0KICAgICAgICAgICAgICAgIHNkLnBvcCgiY29udjEud2VpZ2h0IikgICMg5peg5rOV6YCC")
parts.append("6YWNLOS4ouW8gyjnlKjpmo/mnLrliJ3lp4vljJYpDQogICAgICAgICMgMikg5Yml56a7IGZjL2hlYWQvY2xhc3NpZmllcijmqKHlnovkvJrmm7/mjaLmiJAgMTIg5aS0KQ0KICAgICAgICBmb3IgayBpbiBsaXN0KHNkLmtleXMoKSk6DQogICAgICAgICAgICBpZiBr")
parts.append("LnN0YXJ0c3dpdGgoKCJmYy4iLCAiaGVhZC4iLCAiY2xhc3NpZmllci4iKSk6DQogICAgICAgICAgICAgICAgc2QucG9wKGspDQogICAgICAgIHRyeToNCiAgICAgICAgICAgIG1pc3NpbmcsIHVuZXhwZWN0ZWQgPSBuZXQubG9hZF9zdGF0ZV9kaWN0KHNkLCBzdHJp")
parts.append("Y3Q9RmFsc2UpDQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToNCiAgICAgICAgICAgIHByaW50KGYiW+adg+mHjV0ge2Rlc2N9IOS4jiBSZXNOZXQzNCDnu5PmnoTkuI3ljLnphY06IHtlfSIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICBjb250aW51ZQ0K")
parts.append("ICAgICAgICBleGVtcHQgPSB7ayBmb3IgayBpbiBtaXNzaW5nIGlmIGsuc3RhcnRzd2l0aCgoImNvbnYxLiIsICJmYy4iKSl9DQogICAgICAgIGlmIG5vdCB1bmV4cGVjdGVkIGFuZCBsZW4obWlzc2luZykgPT0gbGVuKGV4ZW1wdCk6DQogICAgICAgICAgICBwcmlu")
parts.append("dChmIlvmnYPph41dIOemu+e6v+mihOiuree7g+WKoOi9veaIkOWKnzoge2Rlc2N9ICjopobnm5Yge2xlbihzZCl9IOS4quWPguaVsOW8oOmHjykiLCBmbHVzaD1UcnVlKQ0KICAgICAgICAgICAgcmV0dXJuIFRydWUsIGRlc2MNCiAgICAgICAgcHJpbnQoZiJb5p2D")
parts.append("6YeNXSB7ZGVzY30g57uT5p6E5LiN5Yy56YWNKG1pc3Npbmcge2xlbihtaXNzaW5nKSAtIGxlbihleGVtcHQpfSDkuKrpnZ7osYHlhY0ga2V5LCAiDQogICAgICAgICAgICAgIGYidW5leHBlY3RlZCB7bGVuKHVuZXhwZWN0ZWQpfSks5bCd6K+V5LiL5LiA5LiqIiwg")
parts.append("Zmx1c2g9VHJ1ZSkNCiAgICByZXR1cm4gRmFsc2UsICIiDQoNCg0KZGVmIF9zY2FuX3BzZXVkbyhiYXNlLCBkZXB0aD0wLCBtYXhfZGVwdGg9Myk6DQogICAgIiIi6YCS5b2S5Zyo5oyC6L2955uu5b2V5LiL5om+6L2v5qCH562+IENTVijpmZDliLbmt7HluqYs6Lez")
parts.append("6L+HIERJQ09NIOebruW9lSnjgIINCg0KICAgIOi/lOWbniAoRGF0YUZyYW1lIOaIliBOb25lLCDmnaXmupDmj4/ov7Ap44CC6K+G5Yir6KeE5YiZOuWQqyBTdHVkeUluc3RhbmNlVUlEIOS4lOiHs+WwkeacieS4gOS4qg0KICAgIHBzZXVkb18g5YmN57yA5YiXKOWm")
parts.append("giBwc2V1ZG9fQUNMKeOAguaJvuS4jeWIsOi/lOWbniAoTm9uZSwgIiIp44CCDQoNCiAgICDor4bliKvmlrnlvI/lj4zpgJrpgZM6DQogICAgMS4g5paH5Lu25ZCN55m95ZCN5Y2VKF9QU0VVRE9fRklMRVMp5b+r6YCf5ZG95LitOw0KICAgIDIuICoq5oyJ5YaF5a65")
parts.append("6K+G5YirKio65Lu75oSPICouY3N2IOWPquimgeihqOWktOWQqyBTdHVkeUluc3RhbmNlVUlEICsgcHNldWRvXyog5YiX5Y2z5ZG95LitDQogICAgICAgKOS4jeS+nei1luaWh+S7tuWQjSznpL7ljLrmlbDmja7pm4bmlLnlkI0v5o2i57uT5p6E5Lmf6IO95omr5Yiw")
parts.append("KeOAgg0KICAgICIiIg0KICAgIGlmIGRlcHRoID4gbWF4X2RlcHRoIG9yIG5vdCBvcy5wYXRoLmlzZGlyKGJhc2UpOg0KICAgICAgICByZXR1cm4gTm9uZSwgIiINCiAgICB0cnk6DQogICAgICAgIGVudHJpZXMgPSBzb3J0ZWQob3MubGlzdGRpcihiYXNlKSkNCiAg")
parts.append("ICBleGNlcHQgT1NFcnJvcjoNCiAgICAgICAgcmV0dXJuIE5vbmUsICIiDQogICAgIyDpgJrpgZMgMTrmlofku7blkI3nmb3lkI3ljZUo5ZCR5ZCO5YW85a65LOivu+WFqOmHjykNCiAgICBmb3IgZm4gaW4gX1BTRVVET19GSUxFUzoNCiAgICAgICAgcCA9IG9zLnBh")
parts.append("dGguam9pbihiYXNlLCBmbikNCiAgICAgICAgaWYgb3MucGF0aC5leGlzdHMocCkgYW5kIG9zLnBhdGguaXNmaWxlKHApOg0KICAgICAgICAgICAgdHJ5Og0KICAgICAgICAgICAgICAgIGNhbmQgPSBwZC5yZWFkX2NzdihwKQ0KICAgICAgICAgICAgZXhjZXB0IEV4")
parts.append("Y2VwdGlvbjoNCiAgICAgICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICAgICAgaWYgIlN0dWR5SW5zdGFuY2VVSUQiIGluIGNhbmQuY29sdW1ucyBhbmQgYW55KA0KICAgICAgICAgICAgICAgICAgICBjLnN0YXJ0c3dpdGgoInBzZXVkb18iKSBmb3IgYyBpbiBj")
parts.append("YW5kLmNvbHVtbnMpOg0KICAgICAgICAgICAgICAgIHJldHVybiBjYW5kLCBvcy5wYXRoLnJlbHBhdGgocCwgIi9rYWdnbGUvaW5wdXQiKQ0KICAgICMg6YCa6YGTIDI65oyJ5YaF5a656K+G5YirKOWPquivu+ihqOWktCBucm93cz0zIOWIpOaWrSzlkb3kuK3lho3l")
parts.append("hajor7spDQogICAgZm9yIGZuIGluIGVudHJpZXM6DQogICAgICAgIGlmIG5vdCBmbi5sb3dlcigpLmVuZHN3aXRoKCIuY3N2Iik6DQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICBwID0gb3MucGF0aC5qb2luKGJhc2UsIGZuKQ0KICAgICAgICBpZiBub3Qg")
parts.append("b3MucGF0aC5pc2ZpbGUocCk6DQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICB0cnk6DQogICAgICAgICAgICBoZWFkID0gcGQucmVhZF9jc3YocCwgbnJvd3M9MykNCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoNCiAgICAgICAgICAgIGNvbnRpbnVlDQog")
parts.append("ICAgICAgIGlmICJTdHVkeUluc3RhbmNlVUlEIiBpbiBoZWFkLmNvbHVtbnMgYW5kIGFueSgNCiAgICAgICAgICAgICAgICBjLnN0YXJ0c3dpdGgoInBzZXVkb18iKSBmb3IgYyBpbiBoZWFkLmNvbHVtbnMpOg0KICAgICAgICAgICAgdHJ5Og0KICAgICAgICAgICAg")
parts.append("ICAgIGNhbmQgPSBwZC5yZWFkX2NzdihwKQ0KICAgICAgICAgICAgICAgIHJldHVybiBjYW5kLCBvcy5wYXRoLnJlbHBhdGgocCwgIi9rYWdnbGUvaW5wdXQiKQ0KICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoNCiAgICAgICAgICAgICAgICBjb250aW51ZQ0K")
parts.append("ICAgIGZvciBzdWIgaW4gZW50cmllczoNCiAgICAgICAgaWYgc3ViIGluIF9TS0lQX0RJUlM6DQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICBjYW5kLCBzcmMgPSBfc2Nhbl9wc2V1ZG8ob3MucGF0aC5qb2luKGJhc2UsIHN1YiksIGRlcHRoICsgMSwgbWF4")
parts.append("X2RlcHRoKQ0KICAgICAgICBpZiBjYW5kIGlzIG5vdCBOb25lOg0KICAgICAgICAgICAgcmV0dXJuIGNhbmQsIHNyYw0KICAgIHJldHVybiBOb25lLCAiIg0KDQoNCmRlZiBmaW5kX3BzZXVkb19kYXRhc2V0KCk6DQogICAgIiIi5ZyoIC9rYWdnbGUvaW5wdXQg5LiL")
parts.append("6YCS5b2S5omr5o+P56S+5Yy66L2v5qCH562+5pWw5o2u6ZuGKExMTSDku47miqXlkYrmj5Dlj5bnmoTkvKrmoIfnrb4p44CCDQoNCiAgICDlhbzlrrnljZXlsYIoL2thZ2dsZS9pbnB1dC88c2x1Zz4vKeS4juWkmuWxgigva2FnZ2xlL2lucHV0L2RhdGFzZXRzLzxz")
parts.append("bHVnPi8p5oyC6L2944CCDQogICAg6L+U5ZueIChEYXRhRnJhbWUg5oiWIE5vbmUsIOadpea6kOaPj+i/sCk75om+5LiN5Yiw6L+U5ZueIChOb25lLCAiIinjgIINCiAgICAiIiINCiAgICBpZiBub3Qgb3MucGF0aC5pc2RpcigiL2thZ2dsZS9pbnB1dCIpOg0KICAg")
parts.append("ICAgICByZXR1cm4gTm9uZSwgIiINCiAgICByZXR1cm4gX3NjYW5fcHNldWRvKCIva2FnZ2xlL2lucHV0IikNCg0KDQpkZWYgbG9hZF9sYWJlbHMocm9vdCwgcHNldWRvX2NvbmY9MC4wLCB1c2VfcmVwb3J0X2xhYmVscz1UcnVlKToNCiAgICAiIiLov5Tlm54gdHJh")
parts.append("aW4gc3R1ZHkg57qn5qCH562+IERhdGFGcmFtZSjooYw9U3R1ZHlJbnN0YW5jZVVJRCwg5YiXPUxBQkVMUywg5YiG5pWwL05hTinjgIINCg0KICAgIOagh+etvuadpea6kOS8mOWFiOe6pygyMDI2LTA5LTA2IOi1tzvml6fnmoQi56S+5Yy66L2v5qCH562+5pWw5o2u")
parts.append("6ZuGIuaWueahiOW3suivgeS8quW8g+eUqOKAlOKAlA0KICAgIGJhcnVuMjEwNCDnmoQgcHNldWRvXyogOTguNyUg5piv5a2X6Z2iIDAuNSDmrbvlgLzjgIHpm7blop7ph4/kv6Hmga8s5pivIGxvc3Mg5oGSIDAuNjkzDQogICAg55qE55yf5qC55ZugKToNCiAgICAx")
parts.append("LiDlrpjmlrkgdHJhaW4uY3N2IOagh+azqCgwLzEs5LuFIDU4LzQ0MDcg5L6LKTsNCiAgICAyLiDlhoXnva7miqXlkYrop4TliJnmir3lj5blmaggcnNuYV9yZXBvcnRfbGFiZWxzKOenu+akjeiHqiByc25hLWtuZWUtYmFzZWxpbmUtdjEsDQogICAgICAg5aSa6K+t")
parts.append("6KiAL+aMieWPpeWQpuWumuaegeaApy/kuKXph43luqbliIbnuqcpOuWvueavj+S7veaUvuWwhOaKpeWRiuS6p+WHuiAxMiDnl4XngbYgc29mdCDliIbmlbANCiAgICAgICAo6Ziz5oCnIDAuNzJ+MC45NSAvIOmYtOaApyAwLjA0fjAuMjAgLyDml6Dor4Hmja49TmFO")
parts.append("KTsNCiAgICAgICDlrpjmlrnnvLrlpLHmoLznlKjmir3lj5bliIbmlbDloavlhYUs5peg6K+B5o2u5qC85L+d5oyBIE5hTijkuI3ov5vmjZ/lpLEs55SxIHZhbGlkIOaOqeeggeaOkumZpCnjgIINCiAgICAiIiINCiAgICB0cmFpbiA9IHBkLnJlYWRfY3N2KG9zLnBh")
parts.append("dGguam9pbihyb290LCAidHJhaW4uY3N2IikpDQogICAgZGYgPSB0cmFpbltbIlN0dWR5SW5zdGFuY2VVSUQiXSArIExBQkVMU10uY29weSgpDQogICAgb2ZmaWNpYWxfY250ID0ge2M6IGludChkZltjXS5ub3RuYSgpLnN1bSgpKSBmb3IgYyBpbiBMQUJFTFN9DQoN")
parts.append("CiAgICAjIEVYUC0wMTM6IG11bHRpLXNvdXJjZSBjb25zZW5zdXMgc29mdCBsYWJlbHMNCiAgICBpZiBvcy5lbnZpcm9uLmdldCgiRVhQMDEzX01VTFRJU09VUkNFIiwgIiIpOg0KICAgICAgICBmcm9tIGV4cDAxM19tdWx0aXNvdXJjZSBpbXBvcnQgYnVpbGRfZXhw")
parts.append("MDEzX2xhYmVscw0KICAgICAgICBvdXQgPSBidWlsZF9leHAwMTNfbGFiZWxzKG9zLnBhdGguam9pbihyb290LCAidHJhaW4uY3N2IiksIHZlcmJvc2U9VHJ1ZSkNCiAgICAgICAgcHJpbnQoIlvmoIfnrb5dIEVYUC0wMTMgbXVsdGktc291cmNlIHRhYmxlIGxvYWRl")
parts.append("ZCIpDQogICAgICAgIHJldHVybiBvdXQNCiAgICAjIEVYUC0wMTI6IExMTS1wcmltYXJ5IHNvZnQgbGFiZWxzIChnb2xkIHdpbnM7IHNpbGVuY2UgMC4yNS8wLjUg4oaSIE5hTikNCiAgICBleHAwMTIgPSBvcy5lbnZpcm9uLmdldCgiRVhQMDEyX0xMTV9QUklNQVJZ")
parts.append("IiwgIiIpDQogICAgaWYgZXhwMDEyOg0KICAgICAgICBmcm9tIGV4cDAxMl9sbG1fcHJpbWFyeSBpbXBvcnQgYnVpbGRfZXhwMDEyX2xhYmVscw0KICAgICAgICBvdXQgPSBidWlsZF9leHAwMTJfbGFiZWxzKG9zLnBhdGguam9pbihyb290LCAidHJhaW4uY3N2Iiks")
parts.append("IHZlcmJvc2U9VHJ1ZSkNCiAgICAgICAgcHJpbnQoIlvmoIfnrb5dIEVYUC0wMTIgTExNLXByaW1hcnkgdGFibGUgbG9hZGVkIikNCiAgICAgICAgcmV0dXJuIG91dA0KDQogICAgaWYgbm90IHVzZV9yZXBvcnRfbGFiZWxzOg0KICAgICAgICBwcmludCgiW+agh+et")
parts.append("vl0gLS1uby1yZXBvcnQtbGFiZWxzOuS7heWumOaWueagh+azqOWPr+eUqCIpDQogICAgICAgIHJldHVybiBkZg0KICAgIGlmIG5vdCBIQVNfUkw6DQogICAgICAgIHByaW50KCJb5qCH562+XSDmnKrmib7liLAgcnNuYV9yZXBvcnRfbGFiZWxzLnB5KOmcgOS4jiBy")
parts.append("c25hX3YzLnB5IOWQjOebruW9lSksIg0KICAgICAgICAgICAgICAi5LuF5a6Y5pa55qCH5rOo5Y+v55SoIikNCiAgICAgICAgcmV0dXJuIGRmDQogICAgaWYgIlJlcG9ydCIgbm90IGluIHRyYWluLmNvbHVtbnM6DQogICAgICAgIHByaW50KCJb5qCH562+XSB0cmFp")
parts.append("bi5jc3Yg5pegIFJlcG9ydCDliJcs5LuF5a6Y5pa55qCH5rOo5Y+v55SoIikNCiAgICAgICAgcmV0dXJuIGRmDQoNCiAgICBwc2V1ZG8gPSBSTC5idWlsZF9wc2V1ZG9fdGFibGUodHJhaW4sIHZlcmJvc2U9VHJ1ZSkNCiAgICBwc2V1ZG8gPSBwc2V1ZG8uc2V0X2lu")
parts.append("ZGV4KCJTdHVkeUluc3RhbmNlVUlEIikNCiAgICBwcmludChmIlvmoIfnrb5dIOWhq+WFhee7n+iuoSh7bGVuKGRmKX0g5LiqIHN0dWR5KToiKQ0KICAgIGZvciBjIGluIExBQkVMUzoNCiAgICAgICAgbWFzayA9IGRmW2NdLmlzbmEoKS52YWx1ZXMNCiAgICAgICAg")
parts.append("aWYgbm90IG1hc2suYW55KCk6DQogICAgICAgICAgICBwcmludChmIiAge2M6MThzfSDlrpjmlrnlhajph48s5peg6ZyA5aGr5YWFIikNCiAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgIHVpZHMgPSBkZi5sb2NbbWFzaywgIlN0dWR5SW5zdGFuY2VVSUQiXQ0K")
parts.append("ICAgICAgICB2YWxzID0gcHNldWRvWyJwc2V1ZG9fIiArIGNdLnJlaW5kZXgodWlkcykudmFsdWVzDQogICAgICAgIG5fZmlsbCA9IGludChwZC5ub3RuYSh2YWxzKS5zdW0oKSkNCiAgICAgICAgZGYubG9jW21hc2ssIGNdID0gdmFscw0KICAgICAgICBwcmludChm")
parts.append("IiAge2M6MThzfSDlrpjmlrkge29mZmljaWFsX2NudFtjXTo0ZH0gKyDmiqXlkYrmir3lj5bloavlhYUge25fZmlsbDo0ZH0iDQogICAgICAgICAgICAgIGYiID0g5pyJ5pWIIHtpbnQoZGZbY10ubm90bmEoKS5zdW0oKSk6NWR9IikNCiAgICByZXR1cm4gZGYNCg0K")
parts.append("DQpfRENNX01JU1NfUFJJTlRFRCA9IFtGYWxzZV0gICMg5Y+q5omT5Y2w5LiA5qyhIERJQ09NIOacquWRveS4reiviuaWrSzpgb/lhY0gMiDkuIcgc2VyaWVzIOWIt+Wxjw0KDQoNCmRlZiByZWFkX2RpY29tX3NsaWNlcyhkY21fZGlyLCBzZXJpZXNfdWlkLCBzdHVk")
parts.append("eV91aWQ9Tm9uZSk6DQogICAgIiIi6K+75Y+W5LiA5LiqIHNlcmllcyDnmoTmiYDmnInliIfniYcs5oyJ54mp55CG5L2N572u5o6S5bqPLOi/lOWbniBbKHBhdGgsIHopXeOAgg0KDQogICAg6Lev5b6E5YW85a655aSa56eNIEthZ2dsZSDluIPlsYAo5a6e5rWLIFJT")
parts.append("TkEgMjAyNiDkuLrkuKTlsYIpOg0KICAgICAgMSkgPGRjbV9kaXI+LzxzZXJpZXNfdWlkPi8gICAgICAgICAgICAgICAgKOaXp+WBh+iuvjpzZXJpZXMg55uu5b2V55u05o6l5oyCKQ0KICAgICAgMikgPGRjbV9kaXI+LzxzdHVkeV91aWQ+LzxzZXJpZXNfdWlkPi8g")
parts.append("ICAgKFJTTkEg5qCH5YeGOnN0dWR5IOebruW9leS4i+WGjeaMgiBzZXJpZXMpDQogICAg5om+5LiN5Yiw5oiW6K+75LiN5Yiw6L+U5ZueIFtd44CCDQogICAgIiIiDQogICAgY2FuZHMgPSBbb3MucGF0aC5qb2luKGRjbV9kaXIsIHNlcmllc191aWQpXQ0KICAgIGlm")
parts.append("IHN0dWR5X3VpZDoNCiAgICAgICAgY2FuZHMuYXBwZW5kKG9zLnBhdGguam9pbihkY21fZGlyLCBzdHVkeV91aWQsIHNlcmllc191aWQpKQ0KICAgIHNkaXIgPSBuZXh0KChjIGZvciBjIGluIGNhbmRzIGlmIG9zLnBhdGguaXNkaXIoYykpLCBOb25lKQ0KICAgIGlm")
parts.append("IHNkaXIgaXMgTm9uZToNCiAgICAgICAgaWYgbm90IF9EQ01fTUlTU19QUklOVEVEWzBdOg0KICAgICAgICAgICAgX0RDTV9NSVNTX1BSSU5URURbMF0gPSBUcnVlDQogICAgICAgICAgICBwcmludChmIlvor4rmlq0tRElDT01dIOaJvuS4jeWIsCBzZXJpZXMg55uu")
parts.append("5b2VLOWwneivlei/hzoge2NhbmRzfSIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICBwcmludChmIlvor4rmlq0tRElDT01dIHtkY21fZGlyfSDmmK/lkKblrZjlnKg6IHtvcy5wYXRoLmlzZGlyKGRjbV9kaXIpfSIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICB0")
parts.append("cnk6DQogICAgICAgICAgICAgICAgaW5uZXIgPSBzb3J0ZWQob3MubGlzdGRpcihkY21fZGlyKSlbOjhdIGlmIG9zLnBhdGguaXNkaXIoZGNtX2RpcikgZWxzZSBbXQ0KICAgICAgICAgICAgICAgIHByaW50KGYiW+iviuaWrS1ESUNPTV0ge2RjbV9kaXJ9IOWunumZ")
parts.append("heWGheWuueagt+acrDoge2lubmVyfSIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICBleGNlcHQgT1NFcnJvcjoNCiAgICAgICAgICAgICAgICBwYXNzDQogICAgICAgIHJldHVybiBbXQ0KICAgIGZpbGVzID0gW29zLnBhdGguam9pbihzZGlyLCBmKSBmb3IgZiBp")
parts.append("biBvcy5saXN0ZGlyKHNkaXIpIGlmIGYubG93ZXIoKS5lbmRzd2l0aCgiLmRjbSIpXQ0KICAgIGlmIG5vdCBmaWxlczoNCiAgICAgICAgaWYgbm90IF9EQ01fTUlTU19QUklOVEVEWzBdOg0KICAgICAgICAgICAgX0RDTV9NSVNTX1BSSU5URURbMF0gPSBUcnVlDQog")
parts.append("ICAgICAgICAgICBwcmludChmIlvor4rmlq0tRElDT01dIOebruW9leWtmOWcqOS9huaXoCAuZGNtIOaWh+S7tjoge3NkaXJ9IiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgICAgIHByaW50KGYiW+iviuaWrS1ESUNPTV0g55uu5b2V5YaF5a655qC35pysOiB7c29ydGVk")
parts.append("KG9zLmxpc3RkaXIoc2RpcikpWzo4XX0iLCBmbHVzaD1UcnVlKQ0KICAgICAgICByZXR1cm4gW10NCiAgICByb3dzID0gW10NCiAgICBmb3IgZiBpbiBmaWxlczoNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgZCA9IHB5ZGljb20uZGNtcmVhZChmLCBzdG9wX2Jl")
parts.append("Zm9yZV9waXhlbHM9RmFsc2UpDQogICAgICAgICAgICBwb3MgPSBkLmdldCgiSW1hZ2VQb3NpdGlvblBhdGllbnQiKQ0KICAgICAgICAgICAgeiA9IGZsb2F0KHBvc1syXSkgaWYgcG9zIGlzIG5vdCBOb25lIGFuZCBsZW4ocG9zKSA+PSAzIGVsc2UgTm9uZQ0KICAg")
parts.append("ICAgICAgICAgcm93cy5hcHBlbmQoKGYsIHopKQ0KICAgICAgICBleGNlcHQgRXhjZXB0aW9uOg0KICAgICAgICAgICAgY29udGludWUNCiAgICByb3dzLnNvcnQoa2V5PWxhbWJkYSByOiAoclsxXSBpcyBOb25lLCByWzFdIGlmIHJbMV0gaXMgbm90IE5vbmUgZWxz")
parts.append("ZSAwLjApKQ0KICAgIHJldHVybiByb3dzDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tIERJQ09NIC0+IDIuNUQgY2xpcCDpooTlpITnkIYgLS0tLS0tLS0tLS0tLS0tLQ0KDQpkZWYgd2luZG93X2FuZF9yZXNpemUoZGNtLCBpbWdfc2l6ZT1JTUdfU0laRSk6DQogICAg")
parts.append("IiIi5Y2V5byg5YiH54mHOlJlc2NhbGUg5ZCO55m+5YiG5L2N56qX5Y+j5YyWIC0+IFswLDFdIC0+IHJlc2l6ZSAyMjTjgIIiIiINCiAgICBhcnIgPSBkY20ucGl4ZWxfYXJyYXkuYXN0eXBlKG5wLmZsb2F0MzIpDQogICAgc2xvcGUgPSBmbG9hdChkY20uZ2V0KCJS")
parts.append("ZXNjYWxlU2xvcGUiLCAxKSkNCiAgICBpbnRlciA9IGZsb2F0KGRjbS5nZXQoIlJlc2NhbGVJbnRlcmNlcHQiLCAwKSkNCiAgICBhcnIgPSBhcnIgKiBzbG9wZSArIGludGVyDQogICAgaWYgZGNtLmdldCgiUGhvdG9tZXRyaWNJbnRlcnByZXRhdGlvbiIpID09ICJN")
parts.append("T05PQ0hST01FMSI6DQogICAgICAgIGFyciA9IGFyci5tYXgoKSAtIGFycg0KICAgIGxvLCBoaSA9IG5wLnBlcmNlbnRpbGUoYXJyLCBbMSwgOTldKQ0KICAgIGFyciA9IG5wLmNsaXAoKGFyciAtIGxvKSAvIG1heChoaSAtIGxvLCAxZS02KSwgMCwgMSkNCiAgICBm")
parts.append("cm9tIFBJTCBpbXBvcnQgSW1hZ2UNCiAgICBpbWcgPSBJbWFnZS5mcm9tYXJyYXkoKGFyciAqIDI1NSkuYXN0eXBlKG5wLnVpbnQ4KSkucmVzaXplKChpbWdfc2l6ZSwgaW1nX3NpemUpLCBJbWFnZS5CSUxJTkVBUikNCiAgICByZXR1cm4gbnAuYXNhcnJheShpbWcs")
parts.append("IGR0eXBlPW5wLmZsb2F0MzIpIC8gMjU1LjANCg0KDQpkZWYgbWFrZV9jbGlwcyhzZXJpZXNfcm93cywgbl9jbGlwcz0yLCBpbWdfc2l6ZT1JTUdfU0laRSk6DQogICAgIiIi5LuO5o6S5bqP5aW955qE5YiH54mH5YiX6KGo55Sf5oiQ6Iul5bmyIDMg6YCa6YGTIGNs")
parts.append("aXAo55u46YK75LiJ5bygKeOAgiIiIg0KICAgIGNsaXBzID0gW10NCiAgICBwYXRocyA9IFtyWzBdIGZvciByIGluIHNlcmllc19yb3dzXQ0KICAgIGlmIGxlbihwYXRocykgPCBDTElQOg0KICAgICAgICByZXR1cm4gY2xpcHMNCiAgICBzdGVwID0gbWF4KDEsIChs")
parts.append("ZW4ocGF0aHMpIC0gMikgLy8gbl9jbGlwcykNCiAgICBmb3IgYyBpbiByYW5nZSgxLCBsZW4ocGF0aHMpIC0gMSwgc3RlcCk6DQogICAgICAgIGlmIGxlbihjbGlwcykgPj0gbl9jbGlwczoNCiAgICAgICAgICAgIGJyZWFrDQogICAgICAgIGNoYW4gPSBbXQ0KICAg")
parts.append("ICAgICBvayA9IFRydWUNCiAgICAgICAgZm9yIGsgaW4gKGMgLSAxLCBjLCBjICsgMSk6DQogICAgICAgICAgICB0cnk6DQogICAgICAgICAgICAgICAgZGNtID0gcHlkaWNvbS5kY21yZWFkKHBhdGhzW2tdKQ0KICAgICAgICAgICAgICAgIGNoYW4uYXBwZW5kKHdp")
parts.append("bmRvd19hbmRfcmVzaXplKGRjbSwgaW1nX3NpemUpKQ0KICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoNCiAgICAgICAgICAgICAgICBvayA9IEZhbHNlDQogICAgICAgICAgICAgICAgYnJlYWsNCiAgICAgICAgaWYgb2s6DQogICAgICAgICAgICBjbGlwcy5h")
parts.append("cHBlbmQobnAuc3RhY2soY2hhbiwgYXhpcz0wKSkgICMgKDMsIEgsIFcpDQogICAgcmV0dXJuIGNsaXBzDQoNCg0KaWYgSEFTX1RPUkNIOg0KICAgICMgLS0tLS0tLS0tLS0tLS0tLSDmlbDmja7pm4YodjI65oyJ5bmz6Z2i6L+H5rukIHNlcmllcykgLS0tLS0tLS0t")
parts.append("LS0tLS0tLQ0KDQogICAgY2xhc3MgS25lZURhdGFzZXQoRGF0YXNldCk6DQogICAgICAgICIiInN0dWR5IOe6p+aVsOaNrumbhizlj6rlj5bmjIflrpogQW5hdG9taWNhbF9QbGFuZSDnmoQgc2VyaWVzLOaLvOaIkCAoQyozLCBILCBXKeOAgiIiIg0KDQogICAgICAg")
parts.append("IGRlZiBfX2luaXRfXyhzZWxmLCBzdHVkaWVzLCBsYWJlbHNfZGYsIGRjbV9yb290LCBzZXJpZXNfZGYsIHBsYW5lLA0KICAgICAgICAgICAgICAgICAgICAgaGFzX3RhcmdldHM9VHJ1ZSwgbl9jbGlwcz0yKToNCiAgICAgICAgICAgIHNlbGYuc3R1ZGllcyA9IHN0")
parts.append("dWRpZXMNCiAgICAgICAgICAgIHNlbGYubGFiZWxzX2RmID0gbGFiZWxzX2RmDQogICAgICAgICAgICBzZWxmLmRjbV9yb290ID0gZGNtX3Jvb3QNCiAgICAgICAgICAgIHNlbGYuc2VyaWVzX2RmID0gc2VyaWVzX2RmDQogICAgICAgICAgICBzZWxmLnBsYW5lID0g")
parts.append("cGxhbmUNCiAgICAgICAgICAgIHNlbGYuaGFzX3RhcmdldHMgPSBoYXNfdGFyZ2V0cw0KICAgICAgICAgICAgc2VsZi5uX2NsaXBzID0gbl9jbGlwcw0KICAgICAgICAgICAgc2VsZi5jYWNoZSA9IHt9ICAjIOavj+aKmOeLrOeri+e8k+WtmDpmb2xkIOe7k+adn+ma")
parts.append("jyBkYXRhc2V0IOmHiuaUvizpgb/lhY0gNSDmipjntK/np68gT09NDQogICAgICAgICAgICAjIOiviuaWreiuoeaVsDror7vliLAgY2xpcCDnmoQgc3R1ZHkgLyDnqbogc3R1ZHkgLyDntK/orqEgY2xpcCDmlbANCiAgICAgICAgICAgIHNlbGYubl9oaXQgPSAwDQog")
parts.append("ICAgICAgICAgICBzZWxmLm5fZW1wdHkgPSAwDQogICAgICAgICAgICBzZWxmLm5fY2xpcHNfdG90YWwgPSAwDQoNCiAgICAgICAgZGVmIHN0YXRzKHNlbGYpOg0KICAgICAgICAgICAgIiIi6L+U5ZueICjor7vliLAgY2xpcCDnmoQgc3R1ZHkg5pWwLCDnqbogc3R1")
parts.append("ZHkg5pWwLCDlubPlnYcgY2xpcCDmlbApLOmYsuatoumdmem7mOeZveiuree7g+OAgiIiIg0KICAgICAgICAgICAgdG90YWwgPSBzZWxmLm5faGl0ICsgc2VsZi5uX2VtcHR5DQogICAgICAgICAgICBhdmcgPSAoc2VsZi5uX2NsaXBzX3RvdGFsIC8gc2VsZi5uX2hp")
parts.append("dCkgaWYgc2VsZi5uX2hpdCBlbHNlIDAuMA0KICAgICAgICAgICAgcmV0dXJuIHNlbGYubl9oaXQsIHNlbGYubl9lbXB0eSwgYXZnDQoNCiAgICAgICAgZGVmIF9sb2FkX3N0dWR5KHNlbGYsIHVpZCk6DQogICAgICAgICAgICBrZXkgPSAodWlkLCBzZWxmLnBsYW5l")
parts.append("KQ0KICAgICAgICAgICAgaWYga2V5IGluIHNlbGYuY2FjaGU6DQogICAgICAgICAgICAgICAgcmV0dXJuIHNlbGYuY2FjaGVba2V5XQ0KICAgICAgICAgICAgc3Jvd3MgPSBzZWxmLnNlcmllc19kZltzZWxmLnNlcmllc19kZlsiU3R1ZHlJbnN0YW5jZVVJRCJdID09")
parts.append("IHVpZF0NCiAgICAgICAgICAgIGlmICJBbmF0b21pY2FsX1BsYW5lIiBpbiBzcm93cy5jb2x1bW5zOg0KICAgICAgICAgICAgICAgIHNyb3dzID0gc3Jvd3Nbc3Jvd3NbIkFuYXRvbWljYWxfUGxhbmUiXSA9PSBzZWxmLnBsYW5lXQ0KICAgICAgICAgICAgICAgIHNy")
parts.append("b3dzID0gc3Jvd3Muc29ydF92YWx1ZXMoIkZsdWlkX1NlbnNpdGl2ZSIsIGtpbmQ9InN0YWJsZSIpLmRyb3BfZHVwbGljYXRlcygNCiAgICAgICAgICAgICAgICAgICAgc3Vic2V0PVsiRmx1aWRfU2Vuc2l0aXZlIl0pLmhlYWQoTUFYX1NFUklFUykNCiAgICAgICAg")
parts.append("ICAgIGVsc2U6DQogICAgICAgICAgICAgICAgc3Jvd3MgPSBzcm93cy5oZWFkKE1BWF9TRVJJRVMpDQogICAgICAgICAgICBjbGlwcyA9IFtdDQogICAgICAgICAgICBmb3IgXywgc3IgaW4gc3Jvd3MuaXRlcnJvd3MoKToNCiAgICAgICAgICAgICAgICBzaWQgPSBz")
parts.append("clsiU2VyaWVzSW5zdGFuY2VVSUQiXQ0KICAgICAgICAgICAgICAgIHJvd3MgPSByZWFkX2RpY29tX3NsaWNlcyhzZWxmLmRjbV9yb290LCBzaWQsIHN0dWR5X3VpZD11aWQpDQogICAgICAgICAgICAgICAgY2xpcHMuZXh0ZW5kKG1ha2VfY2xpcHMocm93cywgbl9j")
parts.append("bGlwcz1zZWxmLm5fY2xpcHMpKQ0KICAgICAgICAgICAgc2VsZi5jYWNoZVt1aWRdID0gY2xpcHMNCiAgICAgICAgICAgICMg5LuF6aaW5qyh5Yqg6L296K6h5pWwKOe8k+WtmOWRveS4reS4jeWGjee0r+WKoCzpgb/lhY3lpJogZXBvY2gg6YeN5aSNKQ0KICAgICAg")
parts.append("ICAgICAgaWYgY2xpcHM6DQogICAgICAgICAgICAgICAgc2VsZi5uX2hpdCArPSAxDQogICAgICAgICAgICAgICAgc2VsZi5uX2NsaXBzX3RvdGFsICs9IGxlbihjbGlwcykNCiAgICAgICAgICAgIGVsc2U6DQogICAgICAgICAgICAgICAgc2VsZi5uX2VtcHR5ICs9")
parts.append("IDENCiAgICAgICAgICAgIHJldHVybiBjbGlwcw0KDQogICAgICAgIGRlZiBfX2xlbl9fKHNlbGYpOg0KICAgICAgICAgICAgcmV0dXJuIGxlbihzZWxmLnN0dWRpZXMpDQoNCiAgICAgICAgZGVmIF9fZ2V0aXRlbV9fKHNlbGYsIGkpOg0KICAgICAgICAgICAgdWlk")
parts.append("ID0gc2VsZi5zdHVkaWVzW2ldDQogICAgICAgICAgICBjbGlwcyA9IHNlbGYuX2xvYWRfc3R1ZHkodWlkKSAgIyDlkb3kuK0v56m66K6h5pWw5ZyoIF9sb2FkX3N0dWR5IOmmluasoeWKoOi9veaXtuWujOaIkA0KICAgICAgICAgICAgaW5fY2ggPSBzZWxmLm5fY2xp")
parts.append("cHMgKiBDTElQICAjIOi+k+WFpemAmumBk+aVsDpjbGlwIOaVsCDDlyDmr48gY2xpcCAzIOWIh+eJhw0KICAgICAgICAgICAgaWYgbm90IGNsaXBzOg0KICAgICAgICAgICAgICAgIHggPSBucC56ZXJvcygoaW5fY2gsIElNR19TSVpFLCBJTUdfU0laRSksIGR0eXBl")
parts.append("PW5wLmZsb2F0MzIpDQogICAgICAgICAgICBlbHNlOg0KICAgICAgICAgICAgICAgIHggPSBucC5jb25jYXRlbmF0ZShjbGlwc1s6IHNlbGYubl9jbGlwc10sIGF4aXM9MCkgICMgKGsqMywgSCwgVykNCiAgICAgICAgICAgICAgICBwYWQgPSBpbl9jaCAtIHguc2hh")
parts.append("cGVbMF0NCiAgICAgICAgICAgICAgICBpZiBwYWQgPiAwOg0KICAgICAgICAgICAgICAgICAgICB4ID0gbnAuY29uY2F0ZW5hdGUoW3gsIG5wLnplcm9zKChwYWQsIElNR19TSVpFLCBJTUdfU0laRSksIGR0eXBlPW5wLmZsb2F0MzIpXSkNCiAgICAgICAgICAgICAg")
parts.append("ICBlbGlmIHBhZCA8IDA6DQogICAgICAgICAgICAgICAgICAgIHggPSB4WzogaW5fY2hdDQogICAgICAgICAgICB4ID0gdG9yY2gudGVuc29yKHgsIGR0eXBlPXRvcmNoLmZsb2F0MzIpDQogICAgICAgICAgICB5ID0gdG9yY2guemVyb3MobGVuKExBQkVMUyksIGR0")
parts.append("eXBlPXRvcmNoLmZsb2F0MzIpDQogICAgICAgICAgICB2YWxpZCA9IHRvcmNoLnplcm9zKGxlbihMQUJFTFMpLCBkdHlwZT10b3JjaC5mbG9hdDMyKQ0KICAgICAgICAgICAgaWYgc2VsZi5oYXNfdGFyZ2V0czoNCiAgICAgICAgICAgICAgICByb3cgPSBzZWxmLmxh")
parts.append("YmVsc19kZltzZWxmLmxhYmVsc19kZlsiU3R1ZHlJbnN0YW5jZVVJRCJdID09IHVpZF0NCiAgICAgICAgICAgICAgICBpZiBsZW4ocm93KToNCiAgICAgICAgICAgICAgICAgICAgZm9yIGosIGMgaW4gZW51bWVyYXRlKExBQkVMUyk6DQogICAgICAgICAgICAgICAg")
parts.append("ICAgICAgICB2ID0gcm93Lmlsb2NbMF1bY10NCiAgICAgICAgICAgICAgICAgICAgICAgIGlmIHBkLm5vdG5hKHYpOg0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIHlbal0gPSBmbG9hdCh2KQ0KICAgICAgICAgICAgICAgICAgICAgICAgICAgIHZhbGlkW2pd")
parts.append("ID0gMS4wDQogICAgICAgICAgICByZXR1cm4geCwgeSwgdmFsaWQNCg0KDQogICAgIyAtLS0tLS0tLS0tLS0tLS0tIOaooeWeiyAtLS0tLS0tLS0tLS0tLS0tDQoNCiAgICBfUFJFVFJBSU5FRF9UUklFRCA9IFtGYWxzZV0gICMg5qih5Z2X57qn5qCH5b+XOuS4i+i9")
parts.append("veWksei0pei/h+S4gOasoeWwseS4jeWGjeWPjeWkjemHjeivlQ0KDQoNCiAgICBkZWYgX2J1aWxkX3Jlc25ldDM0KHVzZV9wcmV0cmFpbmVkPUZhbHNlLCBvZmZsaW5lX3BhdGhzPU5vbmUpOg0KICAgICAgICAiIiLmnoTpgKAgUmVzTmV0MzQg5bm25Yqg6L29IElt")
parts.append("YWdlTmV0IOmihOiuree7g+adg+mHjeOAgg0KDQogICAgICAgIOadg+mHjeadpea6kOS8mOWFiOe6pzoNCiAgICAgICAgMS4gb2ZmbGluZV9wYXRoczrmjILovb3nmoTnprvnur/mnYPph43mlofku7Yo5omr5o+PIC9rYWdnbGUvaW5wdXQg5b6X5YiwLOaXoOmcgOiB")
parts.append("lOe9kSk7DQogICAgICAgIDIuIHVzZV9wcmV0cmFpbmVkPVRydWU66IGU572R5LiL6L29IHRvcmNodmlzaW9uIOadg+mHjSjluKbph43or5Us5aSx6LSl5Zue6YCA6ZqP5py6KTsNCiAgICAgICAgMy4g5ZCm5YiZ6ZqP5py65Yid5aeL5YyW44CCDQogICAgICAgIOaI")
parts.append("kOWKn+WKoOi9veWQjuadg+mHjeeUsSB0b3JjaCDnvJPlrZgs5ZCO57utIGZvbGQg56eS5Yqg6L2944CCDQogICAgICAgICIiIg0KICAgICAgICBuZXQgPSB0dm0ucmVzbmV0MzQod2VpZ2h0cz1Ob25lKQ0KICAgICAgICBpZiBvZmZsaW5lX3BhdGhzOg0KICAgICAg")
parts.append("ICAgICAgb2ssIGRlc2MgPSBfdHJ5X2xvYWRfb2ZmbGluZShuZXQsIG9mZmxpbmVfcGF0aHMpDQogICAgICAgICAgICBpZiBvazoNCiAgICAgICAgICAgICAgICByZXR1cm4gbmV0DQogICAgICAgICAgICBwcmludChmIlvmnYPph41dIOemu+e6v+WAmemAieWdh+S4")
parts.append("jeWMuemFjSh7ZGVzY30pLOWbnumAgOWIsOmaj+acui/ogZTnvZHot6/lvoQiLCBmbHVzaD1UcnVlKQ0KICAgICAgICBpZiBub3QgdXNlX3ByZXRyYWluZWQ6DQogICAgICAgICAgICBwcmludCgiW+adg+mHjV0g5pyq5byA5ZCvIC0tcHJldHJhaW5lZCDkuJTml6Dn")
parts.append("prvnur/mnYPph40s5L2/55So6ZqP5py65Yid5aeL5YyWIg0KICAgICAgICAgICAgICAgICAgIijkuI3ogZTnvZE75aaC6ZyA6aKE6K6t57uD6K+35YqgIC0tcHJldHJhaW5lZCDlubblvIDlkK8gSW50ZXJuZXQsIg0KICAgICAgICAgICAgICAgICAgIuaIluaMgui9")
parts.append("veemu+e6v+adg+mHjeaVsOaNrumbhikiLCBmbHVzaD1UcnVlKQ0KICAgICAgICAgICAgcmV0dXJuIG5ldA0KICAgICAgICBpZiBfUFJFVFJBSU5FRF9UUklFRFswXToNCiAgICAgICAgICAgIHJldHVybiBuZXQgICMg5LmL5YmN5bey56Gu6K6k5LiL6L295LiN5LqG")
parts.append("LOebtOaOpemaj+acug0KICAgICAgICBmb3IgYXR0ZW1wdCBpbiByYW5nZSgxLCA0KToNCiAgICAgICAgICAgIHRyeToNCiAgICAgICAgICAgICAgICBuZXQgPSB0dm0ucmVzbmV0MzQod2VpZ2h0cz10dm0uUmVzTmV0MzRfV2VpZ2h0cy5JTUFHRU5FVDFLX1YxDQog")
parts.append("ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGlmIGhhc2F0dHIodHZtLCAiUmVzTmV0MzRfV2VpZ2h0cyIpIGVsc2UgTm9uZSkNCiAgICAgICAgICAgICAgICByZXR1cm4gbmV0DQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6DQogICAg")
parts.append("ICAgICAgICAgICAgcHJpbnQoZiJb5p2D6YeNXSBSZXNOZXQzNCDpooTorq3nu4PkuIvovb3lpLHotKUo56ysIHthdHRlbXB0fS8zIOasoSk6IHtlfSIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICAgICAgdGltZS5zbGVlcCg1KQ0KICAgICAgICBfUFJFVFJBSU5F")
parts.append("RF9UUklFRFswXSA9IFRydWUNCiAgICAgICAgcHJpbnQoIlvmnYPph41dIDMg5qyh5Z2H5aSx6LSlLOaUueeUqOmaj+acuuWIneWni+WMlijnsr7luqbkvJrmmI7mmL7kuIvpmY075bu66K6u6YeN6K+V5oiW5oyC6L296aKE6K6t57uD5p2D6YeN5pWw5o2u6ZuGKSIs")
parts.append("IGZsdXNoPVRydWUpDQogICAgICAgIHJldHVybiB0dm0ucmVzbmV0MzQod2VpZ2h0cz1Ob25lKQ0KDQoNCiAgICBjbGFzcyBLbmVlTW9kZWwobm4uTW9kdWxlKToNCiAgICAgICAgIiIiUmVzTmV0MzQg5YWx5LqrIGJhY2tib25lICsg5YWo5bGA5rGg5YyWICsgMTIg")
parts.append("c2lnbW9pZCDlpLQo5q+P5Liq5bmz6Z2i5LiA5Liq5a6e5L6LKeOAgiIiIg0KDQogICAgICAgIGRlZiBfX2luaXRfXyhzZWxmLCBudW1fbGFiZWxzPWxlbihMQUJFTFMpLCBjaGFubmVscz1DTElQLCBwcmV0cmFpbmVkPUZhbHNlLCBvZmZsaW5lX3BhdGhzPU5vbmUp")
parts.append("Og0KICAgICAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpDQogICAgICAgICAgICBpZiBIQVNfVFY6DQogICAgICAgICAgICAgICAgc2VsZi5iYWNrYm9uZSA9IF9idWlsZF9yZXNuZXQzNCh1c2VfcHJldHJhaW5lZD1wcmV0cmFpbmVkLA0KICAgICAgICAgICAgICAg")
parts.append("ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgb2ZmbGluZV9wYXRocz1vZmZsaW5lX3BhdGhzKQ0KICAgICAgICAgICAgZWxzZToNCiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIumcgOimgSB0b3JjaHZpc2lvbijor7flnKggS2FnZ2xl")
parts.append("IE5vdGVib29rIEdQVSDnjq/looPov5DooYwpIikNCiAgICAgICAgICAgIGNvbnYxX3cgPSBzZWxmLmJhY2tib25lLmNvbnYxLndlaWdodCAgIyDkv53nlZnpooTorq3nu4MgY29udjEg5p2D6YeNDQogICAgICAgICAgICBzZWxmLmJhY2tib25lLmNvbnYxID0gbm4u")
parts.append("Q29udjJkKGNoYW5uZWxzLCA2NCwga2VybmVsX3NpemU9Nywgc3RyaWRlPTIsIHBhZGRpbmc9MywgYmlhcz1GYWxzZSkNCiAgICAgICAgICAgIGlmIGNvbnYxX3cuc2hhcGUgPT0gc2VsZi5iYWNrYm9uZS5jb252MS53ZWlnaHQuc2hhcGU6ICAjIGNoYW5uZWxzPT0z")
parts.append("IOaXtuebtOaOpeayv+eUqA0KICAgICAgICAgICAgICAgIHNlbGYuYmFja2JvbmUuY29udjEud2VpZ2h0LmRhdGEuY29weV8oY29udjFfdy5kYXRhKQ0KICAgICAgICAgICAgZWxpZiBjaGFubmVscyAlIDMgPT0gMCBhbmQgY29udjFfdy5zaGFwZVsxXSA9PSAzOg0K")
parts.append("ICAgICAgICAgICAgICAgICMgMi41RCDpgJrpgZPmianlsZU65oqKIDMg6YCa6YGT6aKE6K6t57uD5p2D6YeN5aSN5Yi2IG5fY2xpcHMg5Lu9KOW4uOingeWBmuazlSkNCiAgICAgICAgICAgICAgICByZXAgPSBjaGFubmVscyAvLyAzDQogICAgICAgICAgICAgICAg")
parts.append("c2VsZi5iYWNrYm9uZS5jb252MS53ZWlnaHQuZGF0YS5jb3B5Xyhjb252MV93LnJlcGVhdCgxLCByZXAsIDEsIDEpKQ0KICAgICAgICAgICAgICAgIHByaW50KGYiW+adg+mHjV0gY29udjEg6aKE6K6t57uD5p2D6YeN5aSN5Yi2IHtyZXB9IOS7veaJqeWxleWIsCB7")
parts.append("Y2hhbm5lbHN9IOmAmumBkyIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICBzZWxmLmJhY2tib25lLmZjID0gbm4uSWRlbnRpdHkoKQ0KICAgICAgICAgICAgc2VsZi5oZWFkID0gbm4uTGluZWFyKDUxMiwgbnVtX2xhYmVscykNCg0KICAgICAgICBkZWYgZm9yd2Fy")
parts.append("ZChzZWxmLCB4KToNCiAgICAgICAgICAgIGZlYXQgPSBzZWxmLmJhY2tib25lKHgpICAjIChCLCA1MTIpDQogICAgICAgICAgICByZXR1cm4gc2VsZi5oZWFkKGZlYXQpICAgIyBsb2dpdHMgKEIsIDEyKQ0KDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0tIOiuree7gyAv")
parts.append("IOmqjOivgSAtLS0tLS0tLS0tLS0tLS0tDQoNCmRlZiB0cmFpbl9lcG9jaChtb2RlbCwgbG9hZGVyLCBvcHQsIGNyaXQsIGRldmljZSk6DQogICAgbW9kZWwudHJhaW4oKQ0KICAgIHRvdCwgY250ID0gMC4wLCAwDQogICAgZm9yIHgsIHksIHZhbGlkIGluIGxvYWRl")
parts.append("cjoNCiAgICAgICAgeCwgeSwgdmFsaWQgPSB4LnRvKGRldmljZSksIHkudG8oZGV2aWNlKSwgdmFsaWQudG8oZGV2aWNlKQ0KICAgICAgICBvcHQuemVyb19ncmFkKCkNCiAgICAgICAgb3V0ID0gbW9kZWwoeCkNCiAgICAgICAgaWYgdmFsaWQuc3VtKCkgPT0gMDoN")
parts.append("CiAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgICMgRVhQLTAxMzogY29uZmlkZW5jZS13ZWlnaHQgc29mdCBCQ0Ug4oCUIHcgPSAyKnx5LTAuNXwgKFVOSy9zaWxlbmNlIGFscmVhZHkgTmFOKQ0KICAgICAgICB3ID0gMi4wICogKHkgLSAwLjUpLmFicygpDQog")
parts.append("ICAgICAgIGRlbm9tID0gKHZhbGlkICogdykuc3VtKCkNCiAgICAgICAgbG9zcyA9IChjcml0KG91dCwgeSkgKiB2YWxpZCAqIHcpLnN1bSgpIC8gZGVub20uY2xhbXAobWluPTFlLTYpDQogICAgICAgIGxvc3MuYmFja3dhcmQoKQ0KICAgICAgICBvcHQuc3RlcCgp")
parts.append("DQogICAgICAgIHRvdCArPSBsb3NzLml0ZW0oKSAqIGxlbih4KQ0KICAgICAgICBjbnQgKz0gbGVuKHgpDQogICAgcmV0dXJuIHRvdCAvIG1heChjbnQsIDEpDQoNCg0KZGVmIHByZWRpY3QobW9kZWwsIGxvYWRlciwgZGV2aWNlKToNCiAgICBtb2RlbC5ldmFsKCkN")
parts.append("CiAgICByb3dzID0gW10NCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToNCiAgICAgICAgZm9yIHgsIF8sIF8gaW4gbG9hZGVyOg0KICAgICAgICAgICAgeCA9IHgudG8oZGV2aWNlKQ0KICAgICAgICAgICAgb3V0ID0gdG9yY2guc2lnbW9pZChtb2RlbCh4KSkNCiAg")
parts.append("ICAgICAgICAgICMgLmNwdSgpLnRvbGlzdCgpIOiAjOmdniAuY3B1KCkubnVtcHkoKTp0b3JjaCAyLjIuMCtjdTExOCDnvJbkuo4gbnVtcHkxLngsDQogICAgICAgICAgICAjIOWcqCBLYWdnbGUgbnVtcHkyLjAuMiDkuIsgdG9yY2jihpJudW1weSBpbnRlcm9wIOW0")
parts.append("qTt0b2xpc3Qg57qvIEMrKyDot6/lvoTlronlhajjgIINCiAgICAgICAgICAgIHJvd3MuZXh0ZW5kKG91dC5jcHUoKS50b2xpc3QoKSkNCiAgICByZXR1cm4gbnAuYXNhcnJheShyb3dzLCBkdHlwZT1ucC5mbG9hdDMyKSBpZiByb3dzIGVsc2UgbnAuemVyb3MoKDAs")
parts.append("IGxlbihMQUJFTFMpKSwgZHR5cGU9bnAuZmxvYXQzMikNCg0KDQpkZWYgcnVuX2ZvbGQobW9kZWwsIHRyYWluX2RzLCB2YWxfZHMsIGRldmljZSwgYXJncyk6DQogICAgdHJhaW5fbG9hZGVyID0gRGF0YUxvYWRlcih0cmFpbl9kcywgYmF0Y2hfc2l6ZT1hcmdzLmJh")
parts.append("dGNoLCBzaHVmZmxlPVRydWUsDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICBudW1fd29ya2Vycz1hcmdzLndvcmtlcnMsIGRyb3BfbGFzdD1GYWxzZSkNCiAgICB2YWxfbG9hZGVyID0gRGF0YUxvYWRlcih2YWxfZHMsIGJhdGNoX3NpemU9YXJncy5iYXRj")
parts.append("aCwgc2h1ZmZsZT1GYWxzZSwNCiAgICAgICAgICAgICAgICAgICAgICAgICAgICBudW1fd29ya2Vycz1hcmdzLndvcmtlcnMsIGRyb3BfbGFzdD1GYWxzZSkNCiAgICBvcHQgPSBBZGFtVyhtb2RlbC5wYXJhbWV0ZXJzKCksIGxyPWFyZ3MubHIsIHdlaWdodF9kZWNh")
parts.append("eT0xZS00KQ0KICAgIHNjaGVkID0gQ29zaW5lQW5uZWFsaW5nTFIob3B0LCBUX21heD1hcmdzLmVwb2NocykNCiAgICBjcml0ID0gbm4uQkNFV2l0aExvZ2l0c0xvc3MocmVkdWN0aW9uPSJub25lIikNCiAgICBmb3IgZXAgaW4gcmFuZ2UoYXJncy5lcG9jaHMpOg0K")
parts.append("ICAgICAgICB0MCA9IHRpbWUudGltZSgpDQogICAgICAgIGxvc3MgPSB0cmFpbl9lcG9jaChtb2RlbCwgdHJhaW5fbG9hZGVyLCBvcHQsIGNyaXQsIGRldmljZSkNCiAgICAgICAgc2NoZWQuc3RlcCgpDQogICAgICAgIGlmIGFyZ3MudmVyYm9zZToNCiAgICAgICAg")
parts.append("ICAgIHByaW50KGYiICBlcG9jaCB7ZXAgKyAxfS97YXJncy5lcG9jaHN9IGxvc3Mge2xvc3M6LjRmfSAoe3RpbWUudGltZSgpIC0gdDA6LjBmfXMpIiwgZmx1c2g9VHJ1ZSkNCiAgICB2YWxfcHJlZCA9IHByZWRpY3QobW9kZWwsIHZhbF9sb2FkZXIsIGRldmljZSkN")
parts.append("CiAgICAjIERJQ09NIOWRveS4reiviuaWrTrorq3nu4Mr6aqM6K+B5bey6L+t5Luj5a6M5q+VLOatpOaXtuiuoeaVsOaJjeWHhuehrg0KICAgIHRoLCBlaCwgYXQgPSB0cmFpbl9kcy5zdGF0cygpDQogICAgdmgsIGV2LCBhdiA9IHZhbF9kcy5zdGF0cygpDQogICAg")
parts.append("cHJpbnQoZiJb5pWw5o2uXSB0cmFpbiDlkb3kuK0ge3RofS97bGVuKHRyYWluX2RzKX0gc3R1ZHko56m6IHtlaH0sIOW5s+WdhyB7YXQ6LjFmfSBjbGlwKSB8ICINCiAgICAgICAgICBmInZhbCDlkb3kuK0ge3ZofS97bGVuKHZhbF9kcyl9IHN0dWR5KOepuiB7ZXZ9")
parts.append("KSIsIGZsdXNoPVRydWUpDQogICAgaWYgdGggPT0gMCBvciB2aCA9PSAwOg0KICAgICAgICBwcmludCgiW+itpuWRil0g5pyJ5oqYIERJQ09NIOWFqOmDqOayoeivu+WIsCjlkb3kuK0gMCnigJTigJTovpPlhaXlsIbmmK/lhajpm7Ys5YiG5pWw5b+F54S2IH4wLjUh")
parts.append("IiwgZmx1c2g9VHJ1ZSkNCiAgICB2YWxfdWlkID0gbnAuYXJyYXkodmFsX2RzLnN0dWRpZXMpICAjIHNodWZmbGU9RmFsc2Us6aG65bqP5LiO5pWw5o2u6ZuG5LiA6Ie0DQogICAgcmV0dXJuIHZhbF9wcmVkLCB2YWxfdWlkDQoNCg0KIyAtLS0tLS0tLS0tLS0tLS0t")
parts.append("IOaOqOeQhiAvIOiejeWQiCAvIOaPkOS6pCAtLS0tLS0tLS0tLS0tLS0tDQoNCmRlZiBwcmVkaWN0X3Rlc3QobW9kZWwsIHRlc3RfZGYsIHNlcmllc19kZiwgcm9vdCwgZGV2aWNlLCBwbGFuZSwgbl9jbGlwcz0yKToNCiAgICAiIiLlr7kgdGVzdCDmr4/kuKogc3R1")
parts.append("ZHkg5oyJ5oyH5a6a5bmz6Z2i5pS26ZuGIGNsaXBzLOmihOa1iyAxMiDkuKrmpoLnjofjgIIiIiINCiAgICBkcyA9IEtuZWVEYXRhc2V0KHRlc3RfZGZbIlN0dWR5SW5zdGFuY2VVSUQiXS50b2xpc3QoKSwgTm9uZSwgcm9vdCwgc2VyaWVzX2RmLA0KICAgICAgICAg")
parts.append("ICAgICAgICAgICAgcGxhbmUsIGhhc190YXJnZXRzPUZhbHNlLCBuX2NsaXBzPW5fY2xpcHMpDQogICAgbG9hZGVyID0gRGF0YUxvYWRlcihkcywgYmF0Y2hfc2l6ZT0xNiwgc2h1ZmZsZT1GYWxzZSwgbnVtX3dvcmtlcnM9MCkNCiAgICByZXR1cm4gcHJlZGljdCht")
parts.append("b2RlbCwgbG9hZGVyLCBkZXZpY2UpDQoNCg0KZGVmIHBsYW5lX29vZl9hdWMob29mLCBsYWJlbHNfZGYsIHVpZF9saXN0KToNCiAgICAiIiLorqHnrpfkuIDkuKrlubPpnaLmqKHlnovlnKjmjIflrpogc3R1ZHkg5a2Q6ZuG5LiK55qE5a6P5bmz5Z2HIE9PRiBBVUMo")
parts.append("5Y+q566X5pyJ5qCH5rOo55qE5YiXKeOAgg0KDQogICAgbGFiZWxzX2RmIOW/hemhu+aYr+OAkOWumOaWuSBnb2xkIOWOn+Wni+ihqOOAkSgwLzEvTmFOLOWmgiB0cmFpbl9jc3Yg5Y6f5YiXKSzkuI3og73kvKANCiAgICBsb2FkX2xhYmVscyDloavlhYXlkI7nmoQg")
parts.append("c29mdCDooajigJTigJRzb2Z0IOS8quagh+etvuaXoumdniBncm91bmQgdHJ1dGgs5Lmf5Lya6K6pDQogICAgcm9jX2F1Y19zY29yZSDltKkoY29udGludW91cynjgII1OCDkvovlsI/moLfmnKzkuIvmn5DliJflj6/og73lj6rmnInljZXkuIDnsbvliKss6Lez6L+H")
parts.append("6K+l5YiX44CCDQogICAgIiIiDQogICAgYXVjcyA9IFtdDQogICAgZm9yIGosIGMgaW4gZW51bWVyYXRlKExBQkVMUyk6DQogICAgICAgIHlfdHJ1ZSA9IGxhYmVsc19kZi5zZXRfaW5kZXgoIlN0dWR5SW5zdGFuY2VVSUQiKVtjXS5yZWluZGV4KHVpZF9saXN0KS52")
parts.append("YWx1ZXMuYXN0eXBlKGZsb2F0KQ0KICAgICAgICB2YWxpZCA9IH5ucC5pc25hbih5X3RydWUpDQogICAgICAgIGlmIHZhbGlkLnN1bSgpID49IDIwOg0KICAgICAgICAgICAgeXYgPSB5X3RydWVbdmFsaWRdDQogICAgICAgICAgICBpZiBsZW4obnAudW5pcXVlKHl2")
parts.append("KSkgPCAyOg0KICAgICAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgICAgICBhdWNzLmFwcGVuZChyb2NfYXVjX3Njb3JlKHl2LCBvb2ZbdmFsaWQsIGpdKSkNCiAgICByZXR1cm4gZmxvYXQobnAubWVhbihhdWNzKSkgaWYgYXVjcyBlbHNlIDAuMA0KDQoNCiMg")
parts.append("LS0tLS0tLS0tLS0tLS0tLSBhcnRpZmFjdCAvIHJlc3VtZSAvIGVuc2VtYmxlIGhlbHBlcnMgLS0tLS0tLS0tLS0tLS0tLQ0KDQpkZWYgX2FydGlmYWN0X3Jvb3QoYXJncyk6DQogICAgYmFzZSA9IFBhdGgoZ2V0YXR0cihhcmdzLCAiYXJ0aWZhY3RfZGlyIiwgIiIp")
parts.append("IG9yICJhcnRpZmFjdHMiKQ0KICAgIGV4cCA9IGdldGF0dHIoYXJncywgImV4cGVyaW1lbnRfaWQiLCAiZXhwXzAwMV81Zm9sZF9zYWdpdHRhbCIpDQogICAgcmV0dXJuIGJhc2UgLyBleHANCg0KDQpkZWYgX2ZvbGRfbWV0YV9wYXRoKGFydGlmYWN0X3Jvb3QsIGZv")
parts.append("bGQpOg0KICAgIHJldHVybiBhcnRpZmFjdF9yb290IC8gZiJmb2xke2ZvbGR9IiAvICJtZXRhLmpzb24iDQoNCg0KZGVmIF9pc19mb2xkX2RvbmUoYXJ0aWZhY3Rfcm9vdCwgZm9sZCk6DQogICAgcCA9IF9mb2xkX21ldGFfcGF0aChhcnRpZmFjdF9yb290LCBmb2xk")
parts.append("KQ0KICAgIGlmIG5vdCBwLmV4aXN0cygpOg0KICAgICAgICByZXR1cm4gRmFsc2UNCiAgICB0cnk6DQogICAgICAgIG1ldGEgPSBqc29uLmxvYWRzKHAucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQ0KICAgICAgICByZXR1cm4gbWV0YS5nZXQoImNvbXBsZXRp")
parts.append("b25fc3RhdHVzIikgPT0gImRvbmUiDQogICAgZXhjZXB0IEV4Y2VwdGlvbjoNCiAgICAgICAgcmV0dXJuIEZhbHNlDQoNCg0KZGVmIF9zYXZlX2ZvbGRfYXJ0aWZhY3QoYXJ0aWZhY3Rfcm9vdCwgZm9sZCwgdGVzdF9wcmVkLCB0ZXN0X3VpZHMsIG1ldGEpOg0KICAg")
parts.append("IGZvbGRfZGlyID0gYXJ0aWZhY3Rfcm9vdCAvIGYiZm9sZHtmb2xkfSINCiAgICBmb2xkX2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpDQogICAgbnAuc2F2ZShzdHIoZm9sZF9kaXIgLyAidGVzdF9wcmVkLm5weSIpLCB0ZXN0X3ByZWQuYXN0")
parts.append("eXBlKG5wLmZsb2F0MzIpKQ0KICAgIHBkLkRhdGFGcmFtZSh7IlN0dWR5SW5zdGFuY2VVSUQiOiB0ZXN0X3VpZHN9KS50b19jc3YoDQogICAgICAgIGZvbGRfZGlyIC8gInRlc3RfdWlkLmNzdiIsIGluZGV4PUZhbHNlDQogICAgKQ0KICAgIG1ldGFbImNvbXBsZXRp")
parts.append("b25fc3RhdHVzIl0gPSAiZG9uZSINCiAgICBtZXRhWyJwcmVkaWN0aW9uX3NoYXBlIl0gPSBsaXN0KHRlc3RfcHJlZC5zaGFwZSkNCiAgICAoZm9sZF9kaXIgLyAibWV0YS5qc29uIikud3JpdGVfdGV4dCgNCiAgICAgICAganNvbi5kdW1wcyhtZXRhLCBlbnN1cmVf")
parts.append("YXNjaWk9RmFsc2UsIGluZGVudD0yKSwgZW5jb2Rpbmc9InV0Zi04Ig0KICAgICkNCg0KDQpkZWYgX2xvYWRfZm9sZF9wcmVkaWN0aW9ucyhhcnRpZmFjdF9yb290LCBmb2xkLCBjYW5vbmljYWxfdWlkcyk6DQogICAgZm9sZF9kaXIgPSBhcnRpZmFjdF9yb290IC8g")
parts.append("ZiJmb2xke2ZvbGR9Ig0KICAgIHByZWQgPSBucC5sb2FkKGZvbGRfZGlyIC8gInRlc3RfcHJlZC5ucHkiKQ0KICAgIHVpZHMgPSBwZC5yZWFkX2Nzdihmb2xkX2RpciAvICJ0ZXN0X3VpZC5jc3YiKVsiU3R1ZHlJbnN0YW5jZVVJRCJdLnRvbGlzdCgpDQogICAgaWYg")
parts.append("cHJlZC5zaGFwZSAhPSAobGVuKGNhbm9uaWNhbF91aWRzKSwgbGVuKExBQkVMUykpOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiZm9sZHtmb2xkfSBzaGFwZSBtaXNtYXRjaDoge3ByZWQuc2hhcGV9IHZzIGV4cGVjdGVkIHsobGVuKGNhbm9uaWNhbF91aWRz")
parts.append("KSwgbGVuKExBQkVMUykpfSIpDQogICAgaWYgdWlkcyAhPSBjYW5vbmljYWxfdWlkczoNCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImZvbGR7Zm9sZH0gVUlEIG9yZGVyIG1pc21hdGNoIikNCiAgICBpZiBub3QgbnAuaXNmaW5pdGUocHJlZCkuYWxsKCk6DQog")
parts.append("ICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJmb2xke2ZvbGR9IGNvbnRhaW5zIE5hTi9JbmYiKQ0KICAgIGlmIHByZWQubWluKCkgPCAwIG9yIHByZWQubWF4KCkgPiAxOg0KICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiZm9sZHtmb2xkfSBwcmVkaWN0aW9ucyBv")
parts.append("dXQgb2YgWzAsMV0iKQ0KICAgIHJldHVybiBwcmVkDQoNCg0KZGVmIF9zYXZlX2V4cGVyaW1lbnRfbWV0YShhcnRpZmFjdF9yb290LCB0cmFpbl9mb2xkX3VpZHMsIHRlc3RfdWlkcywgYXJncyk6DQogICAgbWV0YSA9IHsNCiAgICAgICAgImV4cGVyaW1lbnRfaWQi")
parts.append("OiBnZXRhdHRyKGFyZ3MsICJleHBlcmltZW50X2lkIiwgImV4cF8wMDFfNWZvbGRfc2FnaXR0YWwiKSwNCiAgICAgICAgImdpdF9jb21taXQiOiBfX2dpdF9jb21taXQoKSwNCiAgICAgICAgImFyZ3MiOiB2YXJzKGFyZ3MpLA0KICAgICAgICAibGFiZWxzIjogTEFC")
parts.append("RUxTLA0KICAgICAgICAidHJhaW5fZm9sZDBfdWlkcyI6IHRyYWluX2ZvbGRfdWlkcywNCiAgICAgICAgInRlc3RfdWlkcyI6IHRlc3RfdWlkcywNCiAgICB9DQogICAgKGFydGlmYWN0X3Jvb3QgLyAiY29uZmlnLmpzb24iKS53cml0ZV90ZXh0KA0KICAgICAgICBq")
parts.append("c29uLmR1bXBzKG1ldGEsIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiDQogICAgKQ0KICAgIHBkLkRhdGFGcmFtZSh7IlN0dWR5SW5zdGFuY2VVSUQiOiB0ZXN0X3VpZHN9KS50b19jc3YoDQogICAgICAgIGFydGlmYWN0X3Jv")
parts.append("b3QgLyAidGVzdF91aWQuY3N2IiwgaW5kZXg9RmFsc2UNCiAgICApDQogICAgZm9sZF9yb3dzID0gW10NCiAgICBmb3IgayBpbiByYW5nZShhcmdzLmZvbGRzKToNCiAgICAgICAgZm9sZF9yb3dzLmFwcGVuZCh7ImZvbGQiOiBrLCAidHJhaW5fdWlkcyI6IHRyYWlu")
parts.append("X2ZvbGRfdWlkc1trXX0pDQogICAgcGQuRGF0YUZyYW1lKGZvbGRfcm93cykudG9fY3N2KGFydGlmYWN0X3Jvb3QgLyAiZm9sZF9hc3NpZ25tZW50LmNzdiIsIGluZGV4PUZhbHNlKQ0KDQoNCmRlZiBfbWF5YmVfa2FnZ2xlX2RhdGFzZXRfdmVyc2lvbihhcnRpZmFj")
parts.append("dF9yb290LCBtZXNzYWdlKToNCiAgICBpZiBvcy5lbnZpcm9uLmdldCgiS0FHR0xFX0tFUk5FTF9SVU5fVFlQRSIpICE9ICJJbnRlcmFjdGl2ZSI6DQogICAgICAgIHJldHVybg0KICAgIHNsdWcgPSBvcy5lbnZpcm9uLmdldCgiS0FHR0xFX0tFUk5FTF9TTFVHIiwg")
parts.append("InJzbmEtYXJ0aWZhY3RzLXRtcCIpDQogICAgb3duZXIgPSBvcy5lbnZpcm9uLmdldCgiS0FHR0xFX1VTRVJOQU1FIiwgInppeWltaW5nIikNCiAgICBkYXRhc2V0X2lkID0gZiJ7b3duZXJ9L3tzbHVnfSINCiAgICB0cnk6DQogICAgICAgIHN1YnByb2Nlc3MucnVu")
parts.append("KA0KICAgICAgICAgICAgWw0KICAgICAgICAgICAgICAgICJrYWdnbGUiLA0KICAgICAgICAgICAgICAgICJkYXRhc2V0cyIsDQogICAgICAgICAgICAgICAgInZlcnNpb24iLA0KICAgICAgICAgICAgICAgICItcCIsDQogICAgICAgICAgICAgICAgc3RyKGFydGlm")
parts.append("YWN0X3Jvb3QpLA0KICAgICAgICAgICAgICAgICItbSIsDQogICAgICAgICAgICAgICAgbWVzc2FnZSwNCiAgICAgICAgICAgICAgICAiLS1kaXItbW9kZSIsDQogICAgICAgICAgICAgICAgInRhciIsDQogICAgICAgICAgICBdLA0KICAgICAgICAgICAgY2hlY2s9")
parts.append("VHJ1ZSwNCiAgICAgICAgICAgIGNhcHR1cmVfb3V0cHV0PVRydWUsDQogICAgICAgICkNCiAgICAgICAgcHJpbnQoZiJbYXJ0aWZhY3RdIHVwbG9hZGVkIGRhdGFzZXQgdmVyc2lvbjoge2RhdGFzZXRfaWR9IiwgZmx1c2g9VHJ1ZSkNCiAgICBleGNlcHQgRXhjZXB0")
parts.append("aW9uIGFzIGU6DQogICAgICAgIHByaW50KGYiW2FydGlmYWN0XSBkYXRhc2V0IHVwbG9hZCBza2lwcGVkOiB7ZX0iLCBmbHVzaD1UcnVlKQ0KDQoNCmRlZiBfX2dpdF9jb21taXQoKToNCiAgICB0cnk6DQogICAgICAgIGltcG9ydCBzdWJwcm9jZXNzDQogICAgICAg")
parts.append("IHJldHVybiBzdWJwcm9jZXNzLmNoZWNrX291dHB1dCgNCiAgICAgICAgICAgIFsiZ2l0IiwgInJldi1wYXJzZSIsICJIRUFEIl0sIHRleHQ9VHJ1ZQ0KICAgICAgICApLnN0cmlwKCkNCiAgICBleGNlcHQgRXhjZXB0aW9uOg0KICAgICAgICByZXR1cm4gInVua25v")
parts.append("d24iDQoNCg0KZGVmIG1haW4oKToNCiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZXBvY2hzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MikNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tYmF0Y2giLCB0eXBlPWlu")
parts.append("dCwgZGVmYXVsdD04KQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1sciIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9M2UtNCkNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0td29ya2VycyIsIHR5cGU9aW50LCBkZWZhdWx0PTApDQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW4t")
parts.append("Y2xpcHMiLCB0eXBlPWludCwgZGVmYXVsdD0yKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zZWVkIiwgdHlwZT1pbnQsIGRlZmF1bHQ9U0VFRCkNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZm9sZHMiLCB0eXBlPWludCwgZGVmYXVsdD1GT0xEUykNCiAgICBhcC5h")
parts.append("ZGRfYXJndW1lbnQoIi0tb25seS1mb2xkIiwgdHlwZT1pbnQsIGRlZmF1bHQ9LTEsDQogICAgICAgICAgICAgICAgICAgIGhlbHA9IuWPqui3keaMh+WumuaKmCjlhbbkvZnot7Pov4c76buY6K6kIC0xPeWFqOmDqCnjgILmjqLot6/nlKg6LS1mb2xkcyA1IC0tb25s")
parts.append("eS1mb2xkIDAg6K6t56ysIDAg5oqYIg0KICAgICAgICAgICAgICAgICAgICAgICAgICIodHJhaW4gNC81IOS4juato+W8j+WQjOWIhuW4gyksdGVzdCDmjqjnkIbnlKjor6XmipjmqKHlnovlh7rmj5DkuqQiKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1wbGFuZXMi")
parts.append("LCB0eXBlPXN0ciwgZGVmYXVsdD0iLCIuam9pbihQTEFORVMpLA0KICAgICAgICAgICAgICAgICAgICBoZWxwPSLpgJflj7fliIbpmpTnmoTlubPpnaLlrZDpm4Ys5aaCIGF4aWFsLHNhZ2l0dGFsKOWwj+WGmSzkuI3ljLrliIblpKflsI/lhpkpIikNCiAgICBhcC5h")
parts.append("ZGRfYXJndW1lbnQoIi0tZnVzZSIsIHR5cGU9c3RyLCBkZWZhdWx0PSJ3ZWlnaHQiLA0KICAgICAgICAgICAgICAgICAgICBjaG9pY2VzPVsid2VpZ2h0IiwgIm1lYW4iXSwNCiAgICAgICAgICAgICAgICAgICAgaGVscD0i5bmz6Z2i6J6N5ZCI5pa55byPOndlaWdo")
parts.append("dD3mjIkgT09GIEFVQyDliqDmnYM7bWVhbj3nroDljZXlubPlnYciKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1wc2V1ZG8tY29uZiIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4wLA0KICAgICAgICAgICAgICAgICAgICBoZWxwPSLkvKrmoIfnrb7nva7kv6Hluqbo")
parts.append("v4fmu6TpmIjlgLwoMD3kuI3ov4fmu6Q7MC41PeS4ouW8g+e9ruS/oeW6pjwwLjXnmoTkvKrmoIfnrb4pIikNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbm8tcmVwb3J0LWxhYmVscyIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsDQogICAgICAgICAgICAgICAgICAgIGhl")
parts.append("bHA9IuemgeeUqOWGhee9ruaKpeWRiuWFs+mUruivjei9r+agh+etvuWFnOW6lSjpu5jorqTlvIDlkK865pyq5oyC56S+5Yy65pWw5o2u6ZuG5pe2Ig0KICAgICAgICAgICAgICAgICAgICAgICAgICLoh6rliqjku44gNDQwNyDku73mlL7lsITmiqXlkYrnlJ/miJDo")
parts.append("va/moIfnrb4s6Zu25oyC6L29KSIpDQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXZlcmJvc2UiLCBhY3Rpb249InN0b3JlX3RydWUiKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1wcmV0cmFpbmVkIiwgYWN0aW9uPSJzdG9yZV90cnVlIiwNCiAgICAgICAgICAgICAg")
parts.append("ICAgICAgaGVscD0i5bCd6K+V5LiL6L29IEltYWdlTmV0IOmihOiuree7g+adg+mHjSjpnIDlvIDlkK8gSW50ZXJuZXQpO+m7mOiupOmaj+acuuWIneWni+WMluS4jeiBlOe9kSIpDQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW9mZmxpbmUtd2VpZ2h0cyIsIHR5cGU9")
parts.append("c3RyLCBkZWZhdWx0PSIiLA0KICAgICAgICAgICAgICAgICAgICBoZWxwPSLpgJflj7fliIbpmpTnmoTnprvnur/mnYPph40gLnB0aCDot6/lvoQo5LyY5YWI57qn5pyA6auYKTvnlZnnqbrliJnoh6rliqjmiavmj4/mjILovb3nm67lvZUiKQ0KICAgIGFwLmFkZF9h")
parts.append("cmd1bWVudCgiLS1uby1vZmZsaW5lLXNjYW4iLCBhY3Rpb249InN0b3JlX3RydWUiLA0KICAgICAgICAgICAgICAgICAgICBoZWxwPSLnpoHnlKjoh6rliqjmiavmj48gL2thZ2dsZS9pbnB1dCDkuIvnmoTnprvnur/mnYPph43mlofku7YiKQ0KICAgIGFwLmFkZF9h")
parts.append("cmd1bWVudCgiLS1kYXRhIiwgdHlwZT1zdHIsIGRlZmF1bHQ9IiIsDQogICAgICAgICAgICAgICAgICAgIGhlbHA9IuaYvuW8j+aMh+WumuaVsOaNruagueebruW9lSjlpoIga2FnZ2xlaHViLmNvbXBldGl0aW9uX2Rvd25sb2FkIOi/lOWbnueahCBwYXRoKSwiDQog")
parts.append("ICAgICAgICAgICAgICAgICAgICAgICAgIuaMh+WumuWQjui3s+i/h+iHquWKqOafpeaJviIpDQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWFydGlmYWN0LWRpciIsIHR5cGU9c3RyLCBkZWZhdWx0PSJhcnRpZmFjdHMiLA0KICAgICAgICAgICAgICAgICAgICBoZWxw")
parts.append("PSJhcnRpZmFjdCDmoLnnm67lvZUo6buY6K6kIGFydGlmYWN0cykiKQ0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1leHBlcmltZW50LWlkIiwgdHlwZT1zdHIsIGRlZmF1bHQ9ImV4cF8wMDFfNWZvbGRfc2FnaXR0YWwiLA0KICAgICAgICAgICAgICAgICAgICBoZWxw")
parts.append("PSJleHBlcmltZW50IElEKOeUqOS6jiBhcnRpZmFjdCDlrZDnm67lvZUpIikNCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcmVzdW1lLWRpciIsIHR5cGU9c3RyLCBkZWZhdWx0PSIiLA0KICAgICAgICAgICAgICAgICAgICBoZWxwPSLlt7LmnIkgYXJ0aWZhY3Qg55uu")
parts.append("5b2VLOWQr+WKqOaXtuaJq+aPjyBtZXRhLmpzb24g6Lez6L+H5bey5a6M5oiQIGZvbGQiKQ0KICAgIGFyZ3MsIF8gPSBhcC5wYXJzZV9rbm93bl9hcmdzKCkNCg0KICAgIHJlc3VtZV9yb290ID0gUGF0aChhcmdzLnJlc3VtZV9kaXIpIGlmIGFyZ3MucmVzdW1lX2Rp")
parts.append("ciBlbHNlIE5vbmUNCiAgICBpZiByZXN1bWVfcm9vdCBhbmQgYXJncy5hcnRpZmFjdF9kaXIgYW5kIG5vdCBhcmdzLmV4cGVyaW1lbnRfaWQ6DQogICAgICAgIGFyZ3MuZXhwZXJpbWVudF9pZCA9IHJlc3VtZV9yb290Lm5hbWUNCg0KICAgIGlmIG5vdCBIQVNfVE9S")
parts.append("Q0ggb3Igbm90IEhBU19UViBvciBub3QgSEFTX1BZRElDT006DQogICAgICAgIHN5cy5leGl0KCLkvp3otZbnvLrlpLE66K+35ZyoIEthZ2dsZSBOb3RlYm9vayhHUFUp546v5aKD6L+Q6KGMKHRvcmNoL3RvcmNodmlzaW9uL3B5ZGljb20pO+acrOWcsOS7heWPr+WB")
parts.append("muaVsOaNruWxgumqjOivgeOAgiIpDQoNCiAgICBucC5yYW5kb20uc2VlZChhcmdzLnNlZWQpDQogICAgdG9yY2gubWFudWFsX3NlZWQoYXJncy5zZWVkKQ0KDQogICAgaWYgYXJncy5kYXRhOg0KICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhvcy5wYXRoLmpvaW4o")
parts.append("YXJncy5kYXRhLCAidHJhaW4uY3N2IikpOg0KICAgICAgICAgICAgcm9vdCwgb3V0ID0gYXJncy5kYXRhLCBhcmdzLmRhdGENCiAgICAgICAgICAgIHByaW50KGYiW+aVsOaNrl0g5L2/55SoIC0tZGF0YSDmjIflrprnm67lvZU6IHtyb290fSIpDQogICAgICAgIGVs")
parts.append("c2U6DQogICAgICAgICAgICBzeXMuZXhpdChmIi0tZGF0YSDmjIflrprnm67lvZUge2FyZ3MuZGF0YX0g5LiL5rKh5pyJIHRyYWluLmNzdizor7fmo4Dmn6Xot6/lvoQiKQ0KICAgIGVsc2U6DQogICAgICAgIHJvb3QsIG91dCA9IHJlc29sdmVfcGF0aHMoKQ0KICAg")
parts.append("IHByaW50KGYiW+aVsOaNrl0gcm9vdD17cm9vdH0gIG91dD17b3V0fSIpDQoNCiAgICB0cmFpbl9jc3YgPSBwZC5yZWFkX2Nzdihvcy5wYXRoLmpvaW4ocm9vdCwgInRyYWluLmNzdiIpKQ0KICAgIGxhYmVsc19kZiA9IGxvYWRfbGFiZWxzKHJvb3QsIHBzZXVkb19j")
parts.append("b25mPWFyZ3MucHNldWRvX2NvbmYsDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgdXNlX3JlcG9ydF9sYWJlbHM9bm90IGFyZ3Mubm9fcmVwb3J0X2xhYmVscykNCiAgICBzZXJpZXNfZGYgPSBwZC5yZWFkX2Nzdihvcy5wYXRoLmpvaW4ocm9vdCwgInRyYWlu")
parts.append("X3Nlcmllcy5jc3YiKSkNCiAgICB0ZXN0X2RmID0gcGQucmVhZF9jc3Yob3MucGF0aC5qb2luKHJvb3QsICJ0ZXN0LmNzdiIpKQ0KICAgIHRlc3Rfc2VyaWVzID0gcGQucmVhZF9jc3Yob3MucGF0aC5qb2luKHJvb3QsICJ0ZXN0X3Nlcmllcy5jc3YiKSkNCiAgICBk")
parts.append("Y21fdHJhaW4gPSBvcy5wYXRoLmpvaW4ocm9vdCwgInRyYWluX3NlcmllcyIpDQogICAgZGNtX3Rlc3QgPSBvcy5wYXRoLmpvaW4ocm9vdCwgInRlc3Rfc2VyaWVzIikNCiAgICBwcmludChmIlvmlbDmja5dIHRyYWluIHtsZW4odHJhaW5fY3N2KX0gc3R1ZHkgLyB0")
parts.append("ZXN0IHtsZW4odGVzdF9kZil9IHN0dWR5IC8gc2VyaWVzIHtsZW4oc2VyaWVzX2RmKX0iKQ0KICAgIF9wcmludF9kY21fdHJlZShyb290LCAiRElDT00iKSAgIyDmiZPljbAgdHJhaW5fc2VyaWVzIOecn+WunuebruW9lee7k+aehCznoa7orqTluIPlsYANCg0KICAg")
parts.append("IHBsYW5lcyA9IFtwLnN0cmlwKCkudGl0bGUoKSBmb3IgcCBpbiBhcmdzLnBsYW5lcy5zcGxpdCgiLCIpIGlmIHAuc3RyaXAoKV0NCiAgICBwbGFuZXMgPSBbcCBmb3IgcCBpbiBwbGFuZXMgaWYgcCBpbiBQTEFORVNdDQogICAgaWYgbm90IHBsYW5lczoNCiAgICAg")
parts.append("ICAgc3lzLmV4aXQoZiItLXBsYW5lcyDml6DmlYgs5Y+v6YCJOiB7UExBTkVTfSIpDQogICAgcHJpbnQoZiJb5bmz6Z2iXSDkvb/nlKgge3BsYW5lc30iKQ0KDQogICAgZGV2aWNlID0gdG9yY2guZGV2aWNlKCJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJs")
parts.append("ZSgpIGVsc2UgImNwdSIpDQogICAgcHJpbnQoZiJb6K6+5aSHXSB7ZGV2aWNlfSIpDQoNCiAgICAjIOemu+e6v+mihOiuree7g+adg+mHjTrmmL7lvI/mjIflrprkvJjlhYgs5ZCm5YiZ6Ieq5Yqo5omr5o+PIC9rYWdnbGUvaW5wdXQNCiAgICBvZmZsaW5lX3BhdGhz")
parts.append("ID0gW10NCiAgICBpZiBhcmdzLm9mZmxpbmVfd2VpZ2h0cy5zdHJpcCgpOg0KICAgICAgICBmb3IgcCBpbiBbeC5zdHJpcCgpIGZvciB4IGluIGFyZ3Mub2ZmbGluZV93ZWlnaHRzLnNwbGl0KCIsIikgaWYgeC5zdHJpcCgpXToNCiAgICAgICAgICAgIGlmIG9zLnBh")
parts.append("dGguaXNmaWxlKHApOg0KICAgICAgICAgICAgICAgIG9mZmxpbmVfcGF0aHMuYXBwZW5kKChwLCBwKSkNCiAgICAgICAgICAgIGVsc2U6DQogICAgICAgICAgICAgICAgcHJpbnQoZiJb5p2D6YeNXSAtLW9mZmxpbmUtd2VpZ2h0cyDmjIflrprnmoTmlofku7bkuI3l")
parts.append("rZjlnKgs6Lez6L+HOiB7cH0iLCBmbHVzaD1UcnVlKQ0KICAgIGVsaWYgbm90IGFyZ3Mubm9fb2ZmbGluZV9zY2FuOg0KICAgICAgICBvZmZsaW5lX3BhdGhzID0gX3NjYW5fb2ZmbGluZV93ZWlnaHRzKCkNCiAgICAgICAgaWYgb2ZmbGluZV9wYXRoczoNCiAgICAg")
parts.append("ICAgICAgIHByaW50KGYiW+adg+mHjV0g6Ieq5Yqo5omr5o+P5YiwIHtsZW4ob2ZmbGluZV9wYXRocyl9IOS4quemu+e6v+adg+mHjeWAmemAiToiLCBmbHVzaD1UcnVlKQ0KICAgICAgICAgICAgZm9yIHAsIGRlc2MgaW4gb2ZmbGluZV9wYXRoc1s6Nl06DQogICAg")
parts.append("ICAgICAgICAgICAgcHJpbnQoZiIgIHtkZXNjfSIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICBpZiBsZW4ob2ZmbGluZV9wYXRocykgPiA2Og0KICAgICAgICAgICAgICAgIHByaW50KGYiICAuLi4g5YWxIHtsZW4ob2ZmbGluZV9wYXRocyl9IOS4qiIsIGZsdXNo")
parts.append("PVRydWUpDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICBwcmludCgiW+adg+mHjV0g5pyq5Zyo5oyC6L2955uu5b2V5Y+R546w56a757q/5p2D6YeNKOWPr+aMgui9veekvuWMuuaJk+WMheeahCByZXNuZXQzNCDmnYPph43mlbDmja7pm4YpIiwgZmx1c2g9VHJ1")
parts.append("ZSkNCiAgICAgICAgICAgIF9wcmludF9tb3VudF90cmVlKCIva2FnZ2xlL2lucHV0IiwgIuadg+mHjSIpDQoNCiAgICBna2YgPSBHcm91cEtGb2xkKG5fc3BsaXRzPWFyZ3MuZm9sZHMpDQogICAgZ3JvdXBzID0gdHJhaW5fY3N2WyJTdHVkeUluc3RhbmNlVUlEIl0u")
parts.append("dmFsdWVzDQoNCiAgICAjIOavj+S4quW5s+mdoueLrOeri+i3keaKmCzorrDlvZUgT09GIOS4jiB0ZXN0IOmihOa1iw0KICAgIHBsYW5lX29vZiwgcGxhbmVfdGVzdCwgcGxhbmVfYXVjLCBwbGFuZV91aWRzID0ge30sIHt9LCB7fSwge30NCiAgICBmb3IgcGxhbmUg")
parts.append("aW4gcGxhbmVzOg0KICAgICAgICBwcmludChmIlxuIyMjIyMjIyMjIyDlubPpnaI6IHtwbGFuZX0gIyMjIyMjIyMjIyIpDQogICAgICAgIG9vZiA9IG5wLnplcm9zKChsZW4odHJhaW5fY3N2KSwgbGVuKExBQkVMUykpKQ0KICAgICAgICBhcnRpZmFjdF9yb290ID0g")
parts.append("X2FydGlmYWN0X3Jvb3QoYXJncykNCiAgICAgICAgY2Fub25pY2FsX3Rlc3RfdWlkcyA9IHRlc3RfZGZbIlN0dWR5SW5zdGFuY2VVSUQiXS50b2xpc3QoKQ0KICAgICAgICB0cmFpbl9mb2xkX3VpZHMgPSBbXQ0KICAgICAgICBmb3IgaywgKHRyX2lkeCwgdmFfaWR4")
parts.append("KSBpbiBlbnVtZXJhdGUoZ2tmLnNwbGl0KHRyYWluX2NzdiwgZ3JvdXBzPWdyb3VwcykpOg0KICAgICAgICAgICAgdHJfc3R1ZGllcyA9IHRyYWluX2Nzdi5pbG9jW3RyX2lkeF1bIlN0dWR5SW5zdGFuY2VVSUQiXS50b2xpc3QoKQ0KICAgICAgICAgICAgdmFfc3R1")
parts.append("ZGllcyA9IHRyYWluX2Nzdi5pbG9jW3ZhX2lkeF1bIlN0dWR5SW5zdGFuY2VVSUQiXS50b2xpc3QoKQ0KICAgICAgICAgICAgdHJhaW5fZm9sZF91aWRzLmFwcGVuZCh0cl9zdHVkaWVzKQ0KICAgICAgICAgICAgaWYgYXJncy5vbmx5X2ZvbGQgPj0gMCBhbmQgayAh")
parts.append("PSBhcmdzLm9ubHlfZm9sZDoNCiAgICAgICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICAgICAgaWYgcmVzdW1lX3Jvb3QgYW5kIF9pc19mb2xkX2RvbmUoYXJ0aWZhY3Rfcm9vdCwgayk6DQogICAgICAgICAgICAgICAgcHJpbnQoZiJbYXJ0aWZhY3RdIGZvbGQg")
parts.append("e2t9IGRvbmUsIHNraXAiLCBmbHVzaD1UcnVlKQ0KICAgICAgICAgICAgICAgIGNvbnRpbnVlDQogICAgICAgICAgICB0cl9kcyA9IEtuZWVEYXRhc2V0KHRyX3N0dWRpZXMsIGxhYmVsc19kZiwgZGNtX3RyYWluLCBzZXJpZXNfZGYsIHBsYW5lLA0KICAgICAgICAg")
parts.append("ICAgICAgICAgICAgICAgICAgICAgICBuX2NsaXBzPWFyZ3Mubl9jbGlwcykNCiAgICAgICAgICAgIHZhX2RzID0gS25lZURhdGFzZXQodmFfc3R1ZGllcywgbGFiZWxzX2RmLCBkY21fdHJhaW4sIHNlcmllc19kZiwgcGxhbmUsDQogICAgICAgICAgICAgICAgICAg")
parts.append("ICAgICAgICAgICAgIG5fY2xpcHM9YXJncy5uX2NsaXBzKQ0KICAgICAgICAgICAgbW9kZWwgPSBLbmVlTW9kZWwocHJldHJhaW5lZD1hcmdzLnByZXRyYWluZWQsDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICBvZmZsaW5lX3BhdGhzPW9mZmxpbmVfcGF0")
parts.append("aHMsDQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjaGFubmVscz1hcmdzLm5fY2xpcHMgKiBDTElQKS50byhkZXZpY2UpDQogICAgICAgICAgICBwcmludChmIlvmipgge2sgKyAxfS97YXJncy5mb2xkc31dIHtwbGFuZX0gdHJhaW4ge2xlbih0cl9kcyl9")
parts.append("IC8gdmFsIHtsZW4odmFfZHMpfSIsIGZsdXNoPVRydWUpDQogICAgICAgICAgICB0MCA9IHRpbWUudGltZSgpDQogICAgICAgICAgICB2YWxfcHJlZCwgdmFsX3VpZCA9IHJ1bl9mb2xkKG1vZGVsLCB0cl9kcywgdmFfZHMsIGRldmljZSwgYXJncykNCiAgICAgICAg")
parts.append("ICAgIGZvbGRfdHJhaW5fdGltZSA9IHRpbWUudGltZSgpIC0gdDANCiAgICAgICAgICAgIGZvciBqLCB1aWQgaW4gZW51bWVyYXRlKHZhbF91aWQpOg0KICAgICAgICAgICAgICAgIHJvdyA9IG5wLndoZXJlKHRyYWluX2NzdlsiU3R1ZHlJbnN0YW5jZVVJRCJdLnZh")
parts.append("bHVlcyA9PSB1aWQpWzBdDQogICAgICAgICAgICAgICAgaWYgbGVuKHJvdyk6DQogICAgICAgICAgICAgICAgICAgIG9vZltyb3dbMF1dID0gdmFsX3ByZWRbal0NCiAgICAgICAgICAgIHQwID0gdGltZS50aW1lKCkNCiAgICAgICAgICAgIHRlc3RfcHJlZCA9IHBy")
parts.append("ZWRpY3RfdGVzdChtb2RlbCwgdGVzdF9kZiwgdGVzdF9zZXJpZXMsIGRjbV90ZXN0LCBkZXZpY2UsIHBsYW5lLCBuX2NsaXBzPWFyZ3Mubl9jbGlwcykNCiAgICAgICAgICAgIGZvbGRfaW5mZXJlbmNlX3RpbWUgPSB0aW1lLnRpbWUoKSAtIHQwDQogICAgICAgICAg")
parts.append("ICBtZXRhID0gew0KICAgICAgICAgICAgICAgICJmb2xkIjogaywNCiAgICAgICAgICAgICAgICAicGxhbmUiOiBwbGFuZSwNCiAgICAgICAgICAgICAgICAibl9jbGlwcyI6IGFyZ3Mubl9jbGlwcywNCiAgICAgICAgICAgICAgICAiaW1hZ2Vfc2l6ZSI6IElNR19T")
parts.append("SVpFLA0KICAgICAgICAgICAgICAgICJlcG9jaHMiOiBhcmdzLmVwb2NocywNCiAgICAgICAgICAgICAgICAiYmF0Y2hfc2l6ZSI6IGFyZ3MuYmF0Y2gsDQogICAgICAgICAgICAgICAgInNlZWQiOiBhcmdzLnNlZWQsDQogICAgICAgICAgICAgICAgImdpdF9jb21t")
parts.append("aXQiOiBfX2dpdF9jb21taXQoKSwNCiAgICAgICAgICAgICAgICAiZm9sZF90cmFpbl90aW1lX3NlYyI6IGZvbGRfdHJhaW5fdGltZSwNCiAgICAgICAgICAgICAgICAiZm9sZF9pbmZlcmVuY2VfdGltZV9zZWMiOiBmb2xkX2luZmVyZW5jZV90aW1lLA0KICAgICAg")
parts.append("ICAgICAgfQ0KICAgICAgICAgICAgX3NhdmVfZm9sZF9hcnRpZmFjdChhcnRpZmFjdF9yb290LCBrLCB0ZXN0X3ByZWQsIGNhbm9uaWNhbF90ZXN0X3VpZHMsIG1ldGEpDQogICAgICAgICAgICB0cnk6DQogICAgICAgICAgICAgICAgd2RpciA9IGFydGlmYWN0X3Jv")
parts.append("b3QgLyBmImZvbGR7a30iIC8gcGxhbmUNCiAgICAgICAgICAgICAgICB3ZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCiAgICAgICAgICAgICAgICB3cGF0aCA9IHdkaXIgLyAibW9kZWwucHQiDQogICAgICAgICAgICAgICAgdG9yY2guc2F2")
parts.append("ZShtb2RlbC5zdGF0ZV9kaWN0KCksIHN0cih3cGF0aCkpDQogICAgICAgICAgICAgICAgbWV0YTIgPSBkaWN0KG1ldGEpDQogICAgICAgICAgICAgICAgbWV0YTJbImNoYW5uZWxzIl0gPSBpbnQoYXJncy5uX2NsaXBzICogQ0xJUCkNCiAgICAgICAgICAgICAgICBt")
parts.append("ZXRhMlsibnVtX2xhYmVscyJdID0gbGVuKExBQkVMUykNCiAgICAgICAgICAgICAgICAod2RpciAvICJtb2RlbF9tZXRhLmpzb24iKS53cml0ZV90ZXh0KA0KICAgICAgICAgICAgICAgICAgICBqc29uLmR1bXBzKG1ldGEyLCBlbnN1cmVfYXNjaWk9RmFsc2UsIGlu")
parts.append("ZGVudD0yKSwgZW5jb2Rpbmc9InV0Zi04IikNCiAgICAgICAgICAgICAgICBwcmludChmIlt3ZWlnaHRzXSBzYXZlZCB7d3BhdGh9IiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToNCiAgICAgICAgICAgICAgICBwcmludChm")
parts.append("Ilt3ZWlnaHRzXSBzYXZlIGZhaWxlZDoge2V9IiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgICAgIHByaW50KGYiW2FydGlmYWN0XSBzYXZlZCBmb2xke2t9IC0+IHthcnRpZmFjdF9yb290IC8gZidmb2xke2t9J30iLCBmbHVzaD1UcnVlKQ0KICAgICAgICAjIGV4cGVy")
parts.append("aW1lbnQtbGV2ZWwgbWV0YSBvbmNlIHBlciBwbGFuZQ0KICAgICAgICBfc2F2ZV9leHBlcmltZW50X21ldGEoYXJ0aWZhY3Rfcm9vdCwgdHJhaW5fZm9sZF91aWRzLCBjYW5vbmljYWxfdGVzdF91aWRzLCBhcmdzKQ0KICAgICAgICBwcmludChmIlthcnRpZmFjdF0g")
parts.append("c2F2ZWQgZXhwZXJpbWVudCBtZXRhIC0+IHthcnRpZmFjdF9yb290fSIsIGZsdXNoPVRydWUpDQogICAgICAgICMg6K+l5bmz6Z2iIE9PRiBBVUMo5a6P5bmz5Z2HKQ0KICAgICAgICB1aWRfbGlzdCA9IHRyYWluX2NzdlsiU3R1ZHlJbnN0YW5jZVVJRCJdLnRvbGlz")
parts.append("dCgpDQogICAgICAgIGlmIGFyZ3Mub25seV9mb2xkID49IDA6DQogICAgICAgICAgICBwcmludChmIlt7cGxhbmV9XSDmjqLot6/mqKHlvI8oLS1vbmx5LWZvbGQge2FyZ3Mub25seV9mb2xkfSk6T09GIOWNleaKmOS4jeWujOaVtCwiDQogICAgICAgICAgICAgICAg")
parts.append("ICBmIui3s+i/hyBPT0YgQVVDKOadg+mHjeiusCAwPeWdh+etiSk7dGVzdCDmjqjnkIbnlKjor6XmipjmqKHlnosiKQ0KICAgICAgICAgICAgYXVjID0gMC4wDQogICAgICAgIGVsc2U6DQogICAgICAgICAgICBhdWMgPSBwbGFuZV9vb2ZfYXVjKG9vZiwgdHJhaW5f")
parts.append("Y3N2W1siU3R1ZHlJbnN0YW5jZVVJRCJdICsgTEFCRUxTXSwgdWlkX2xpc3QpDQogICAgICAgICAgICBwcmludChmIlt7cGxhbmV9XSBPT0Yg5a6P5bmz5Z2HIEFVQyB7YXVjOi40Zn0iKQ0KICAgICAgICBwbGFuZV9vb2ZbcGxhbmVdLCBwbGFuZV9hdWNbcGxhbmVd")
parts.append("LCBwbGFuZV91aWRzW3BsYW5lXSA9IG9vZiwgYXVjLCB1aWRfbGlzdA0KICAgICAgICAjIGxvYWQgc2F2ZWQgZm9sZCBwcmVkaWN0aW9ucyBmb3IgdGhpcyBwbGFuZQ0KICAgICAgICBwZXJfZm9sZCA9IFtdDQogICAgICAgIG1pc3NpbmcgPSBbXQ0KICAgICAgICBm")
parts.append("b3IgayBpbiByYW5nZShhcmdzLmZvbGRzKToNCiAgICAgICAgICAgIHAgPSBhcnRpZmFjdF9yb290IC8gZiJmb2xke2t9IiAvICJ0ZXN0X3ByZWQubnB5Ig0KICAgICAgICAgICAgaWYgbm90IHAuZXhpc3RzKCk6DQogICAgICAgICAgICAgICAgbWlzc2luZy5hcHBl")
parts.append("bmQoaykNCiAgICAgICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICAgICAgdHJ5Og0KICAgICAgICAgICAgICAgIHBlcl9mb2xkLmFwcGVuZChfbG9hZF9mb2xkX3ByZWRpY3Rpb25zKGFydGlmYWN0X3Jvb3QsIGssIGNhbm9uaWNhbF90ZXN0X3VpZHMpKQ0KICAg")
parts.append("ICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOg0KICAgICAgICAgICAgICAgIG1pc3NpbmcuYXBwZW5kKGspDQogICAgICAgICAgICAgICAgcHJpbnQoZiJbZW5zZW1ibGVdIGZvbGR7a30gaW52YWxpZDoge2V9IiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgaWYg")
parts.append("bGVuKG1pc3NpbmcpID09IGFyZ3MuZm9sZHM6DQogICAgICAgICAgICBzeXMuZXhpdChmIltlbnNlbWJsZV0gbm8gdmFsaWQgZm9sZCBwcmVkaWN0aW9ucyBhdmFpbGFibGUgdW5kZXIge2FydGlmYWN0X3Jvb3R9IikNCiAgICAgICAgaWYgbWlzc2luZzoNCiAgICAg")
parts.append("ICAgICAgIHByaW50KGYiW2Vuc2VtYmxlXSBtaXNzaW5nIGZvbGRzOiB7bWlzc2luZ307IGVuc2VtYmxlIHdpdGggYXZhaWxhYmxlIGZvbGRzIG9ubHkiLCBmbHVzaD1UcnVlKQ0KICAgICAgICBwbGFuZV90ZXN0W3BsYW5lXSA9IG5wLm1lYW4ocGVyX2ZvbGQsIGF4")
parts.append("aXM9MCkgaWYgcGVyX2ZvbGQgZWxzZSBOb25lDQogICAgICAgIGlmIHBsYW5lX3Rlc3RbcGxhbmVdIGlzIG5vdCBOb25lOg0KICAgICAgICAgICAgcHJpbnQoZiJbZW5zZW1ibGVdIHtwbGFuZX0gZW5zZW1ibGVkIGZyb20ge2xlbihwZXJfZm9sZCl9IGZvbGRzLCBz")
parts.append("aGFwZT17cGxhbmVfdGVzdFtwbGFuZV0uc2hhcGV9IikNCg0KICAgICMg6J6N5ZCIDQogICAgaWYgYXJncy5mdXNlID09ICJ3ZWlnaHQiOg0KICAgICAgICB3c3VtID0gc3VtKG1heChhLCAxZS02KSBmb3IgYSBpbiBwbGFuZV9hdWMudmFsdWVzKCkpDQogICAgICAg")
parts.append("IHdlaWdodHMgPSB7cDogbWF4KGEsIDFlLTYpIC8gd3N1bSBmb3IgcCwgYSBpbiBwbGFuZV9hdWMuaXRlbXMoKX0NCiAgICAgICAgcHJpbnQoZiJb6J6N5ZCIXSBPT0YgQVVDIOWKoOadgzogeyB7cDogcm91bmQodywgNCkgZm9yIHAsIHcgaW4gd2VpZ2h0cy5pdGVt")
parts.append("cygpfSB9IikNCiAgICBlbHNlOg0KICAgICAgICB3ZWlnaHRzID0ge3A6IDEuMCAvIGxlbihwbGFuZXMpIGZvciBwIGluIHBsYW5lc30NCiAgICAgICAgcHJpbnQoIlvono3lkIhdIOeugOWNleW5s+WdhyIpDQogICAgZnVzZWQgPSBucC56ZXJvc19saWtlKHBsYW5l")
parts.append("X3Rlc3RbcGxhbmVzWzBdXSkNCiAgICBmb3IgcCwgdyBpbiB3ZWlnaHRzLml0ZW1zKCk6DQogICAgICAgIGZ1c2VkICs9IHcgKiBwbGFuZV90ZXN0W3BdDQogICAgc3ViID0gcGQuRGF0YUZyYW1lKHsiU3R1ZHlJbnN0YW5jZVVJRCI6IHRlc3RfZGZbIlN0dWR5SW5z")
parts.append("dGFuY2VVSUQiXSwgKip7DQogICAgICAgIExBQkVMU1tqXTogZnVzZWRbOiwgal0gZm9yIGogaW4gcmFuZ2UobGVuKExBQkVMUykpfX0pDQogICAgc3ViX3BhdGggPSBvcy5wYXRoLmpvaW4ob3V0LCAic3VibWlzc2lvbi5jc3YiKQ0KICAgIHN1Yi50b19jc3Yoc3Vi")
parts.append("X3BhdGgsIGluZGV4PUZhbHNlKQ0KICAgIHByaW50KGYiXG7mj5DkuqTmlofku7blt7LnlJ/miJA6IHtzdWJfcGF0aH0gIHNoYXBlPXtzdWIuc2hhcGV9IikNCg0KDQppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOg0KICAgIG1haW4oKQ0K")
open('/kaggle/working/rsna_v3.py','wb').write(base64.b64decode(''.join(parts)))
print('wrote rsna_v3.py', os.path.getsize('/kaggle/working/rsna_v3.py'))

import base64, os
# embed exp013_multisource.py
parts=[]
parts.append("IiIiRVhQLTAxMyBtdWx0aS1zb3VyY2Ugc29mdCBsYWJlbHMuCgpTb3VyY2VzIChwcm9iYWJpbGl0eS1zcGFjZSBibGVuZCwgc2lsZW5jZSDihpIgTmFOKToKICAtIHN0ZXZlbmxlZWhhbnMgbGxtX2xhYmVsc192NF9ibGVuZC5jc3YgIChnb2xkLWtleSAwLjg5MykK")
parts.append("ICAtIGxpeGluNzMgbGFiZWxzX2xsbV9ncHQ1NnNvbC5jc3YgICAgICAgICgwLjgzNSwgZGVjb3JyZWxhdGVkKQogIC0gcGlsa3dhbmcgcmVwb3J0X2xhYmVsc192Mi5jc3YgICAgICAgICAgKDAuODcwLCBjb25mICsgVU5LIHZlcmRpY3QpCgpXZWlnaHRzOiBiYXNl")
parts.append("IDAuNSAvIDAuMiAvIDAuMywgdGhlbiBvcHRpb25hbCBwZXItY2xhc3MgcmVsaWFiaWxpdHkgb24gZ29sZDU4LgpTaWxlbmNlOiAwLjI1LzAuNSByb3VuZGVkIGJ1Y2tldHMgYW5kIHBpbGt3YW5nIFVOSyDihpIgTmFOIChuZXZlciB0cmFpbikuClRhcmdldHM6IHNo")
parts.append("cmluayAwLjUrKHAtMC41KSowLjc1LCBjbGlwIFswLjA4LCAwLjkyXS4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFz")
parts.append("IGFzIHBkCgpMQUJFTFMgPSBbCiAgICAiQUNMIiwgIk1DTCIsICJNZWRpYWwgTWVuaXNjdXMiLCAiTGF0ZXJhbCBNZW5pc2N1cyIsCiAgICAiTWVkaWFsIE9BIiwgIkxhdGVyYWwgT0EiLCAiUEYgT0EiLCAiRWZmdXNpb24iLAogICAgIlN5bm92aXRpcyIsICJCYWtl")
parts.append("cidzIiwgIkNvbnR1c2lvbiIsICJGcmFjdHVyZSIsCl0KClNJTEVOQ0VfUk9VTkRFRCA9IHswLjI1LCAwLjV9CkNMSVBfTE8sIENMSVBfSEkgPSAwLjA4LCAwLjkyCkJBU0VfVyA9IHsic3RldmVuIjogMC41MCwgImxpeGluIjogMC4yMCwgInBpbGsiOiAwLjMwfQoK")
parts.append("CmRlZiBfc3RhZ2Vfcm9vdCgpIC0+IFBhdGg6CiAgICByZXR1cm4gUGF0aChvcy5lbnZpcm9uLmdldCgKICAgICAgICAiRVhQMDEzX0xBQkVMX1JPT1QiLAogICAgICAgIHIiRDov5qGM6Z2iL0thZ2dsZS9yc25hX2tuZWUvX2RhdGFzZXRfc3RhZ2UiLAogICAgKSkK")
parts.append("CgpkZWYgX2ZpbmQobmFtZTogc3RyKSAtPiBQYXRoOgogICAgZW52ID0gb3MuZW52aXJvbi5nZXQoZiJFWFAwMTNfe25hbWUudXBwZXIoKX0iKQogICAgaWYgZW52IGFuZCBQYXRoKGVudikuaXNfZmlsZSgpOgogICAgICAgIHJldHVybiBQYXRoKGVudikKICAgIGZv")
parts.append("ciByb290IGluICgKICAgICAgICBQYXRoKCIva2FnZ2xlL2lucHV0IiksCiAgICAgICAgX3N0YWdlX3Jvb3QoKSwKICAgICk6CiAgICAgICAgaWYgbm90IHJvb3QuaXNfZGlyKCk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgZm9yIHAgaW4gcm9vdC5yZ2xv")
parts.append("YihuYW1lKToKICAgICAgICAgICAgcmV0dXJuIHAKICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKG5hbWUpCgoKZGVmIF9jbGVhbihhcnI6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICBvdXQgPSBhcnIuYXN0eXBlKG5wLmZsb2F0NjQpLmNvcHkoKQog")
parts.append("ICAgb3V0ID0gbnAud2hlcmUobnAuaXNpbihucC5yb3VuZChvdXQsIDIpLCBsaXN0KFNJTEVOQ0VfUk9VTkRFRCkpLCBucC5uYW4sIG91dCkKICAgIG91dCA9IG5wLndoZXJlKG5wLmlzZmluaXRlKG91dCksIDAuNSArIChvdXQgLSAwLjUpICogMC43NSwgb3V0KQog")
parts.append("ICAgb3V0ID0gbnAuY2xpcChvdXQsIENMSVBfTE8sIENMSVBfSEkpCiAgICByZXR1cm4gb3V0CgoKZGVmIF9sb2FkX3NvdXJjZXMoKToKICAgIHN0ZXZlbiA9IHBkLnJlYWRfY3N2KF9maW5kKCJsbG1fbGFiZWxzX3Y0X2JsZW5kLmNzdiIpKS5zZXRfaW5kZXgoIlN0")
parts.append("dWR5SW5zdGFuY2VVSUQiKQogICAgbGl4aW4gPSBwZC5yZWFkX2NzdihfZmluZCgibGFiZWxzX2xsbV9ncHQ1NnNvbC5jc3YiKSkuc2V0X2luZGV4KCJTdHVkeUluc3RhbmNlVUlEIikKICAgIHBpbGsgPSBwZC5yZWFkX2NzdihfZmluZCgicmVwb3J0X2xhYmVsc192")
parts.append("Mi5jc3YiKSkuc2V0X2luZGV4KCJTdHVkeUluc3RhbmNlVUlEIikKICAgIHJldHVybiBzdGV2ZW4sIGxpeGluLCBwaWxrCgoKZGVmIF9waWxrX3NvZnQocGlsazogcGQuRGF0YUZyYW1lLCBjOiBzdHIpIC0+IHBkLlNlcmllczoKICAgICIiIk1hcCBwaWxrd2FuZyB2")
parts.append("ZXJkaWN0K2NvbmYgdG8gc29mdCBwcm9iczsgVU5LIOKGkiBOYU4uIiIiCiAgICBpZiBmIntjfV9fdmVyZGljdCIgbm90IGluIHBpbGsuY29sdW1ucyBvciBjIG5vdCBpbiBwaWxrLmNvbHVtbnM6CiAgICAgICAgcmV0dXJuIHBkLlNlcmllcyhucC5uYW4sIGluZGV4")
parts.append("PXBpbGsuaW5kZXgpCiAgICB2ID0gcGlsa1tmIntjfV9fdmVyZGljdCJdLmFzdHlwZShzdHIpLnN0ci51cHBlcigpCiAgICByYXcgPSBwaWxrW2NdLmFzdHlwZShmbG9hdCkKICAgIG91dCA9IHJhdy5jb3B5KCkKICAgIG91dFt2ID09ICJVTksiXSA9IG5wLm5hbgog")
parts.append("ICAgcmV0dXJuIG91dAoKCmRlZiBidWlsZF9leHAwMTNfbGFiZWxzKHRyYWluX2Nzdjogc3RyIHwgUGF0aCwgdmVyYm9zZTogYm9vbCA9IFRydWUpIC0+IHBkLkRhdGFGcmFtZToKICAgIHRyYWluID0gcGQucmVhZF9jc3YodHJhaW5fY3N2KQogICAgZGYgPSB0cmFp")
parts.append("bltbIlN0dWR5SW5zdGFuY2VVSUQiXSArIExBQkVMU10uY29weSgpCiAgICBzdGV2ZW4sIGxpeGluLCBwaWxrID0gX2xvYWRfc291cmNlcygpCiAgICBpZiB2ZXJib3NlOgogICAgICAgIHByaW50KCJbRVhQLTAxM10gc291cmNlczoiLAogICAgICAgICAgICAgIGYi")
parts.append("c3RldmVuPXtsZW4oc3RldmVuKX0gbGl4aW49e2xlbihsaXhpbil9IHBpbGs9e2xlbihwaWxrKX0iKQogICAgICAgIHByaW50KGYiW0VYUC0wMTNdIHdlaWdodHMge0JBU0VfV30iKQoKICAgIG5fY2VsbHMgPSAwCiAgICBmb3IgYyBpbiBMQUJFTFM6CiAgICAgICAg")
parts.append("bWFzayA9IGRmW2NdLmlzbmEoKS52YWx1ZXMKICAgICAgICB1aWRzID0gZGYubG9jW21hc2ssICJTdHVkeUluc3RhbmNlVUlEIl0KCiAgICAgICAgcGFydHMgPSBbXQogICAgICAgIHdzID0gW10KICAgICAgICBmb3Igc3JjLCBrZXksIHcgaW4gKAogICAgICAgICAg")
parts.append("ICAoc3RldmVuLCBjLCBCQVNFX1dbInN0ZXZlbiJdKSwKICAgICAgICAgICAgKGxpeGluLCBjLCBCQVNFX1dbImxpeGluIl0pLAogICAgICAgICAgICAocGlsaywgTm9uZSwgQkFTRV9XWyJwaWxrIl0pLAogICAgICAgICk6CiAgICAgICAgICAgIGlmIGtleSBpcyBu")
parts.append("b3QgTm9uZToKICAgICAgICAgICAgICAgIGlmIGtleSBub3QgaW4gc3JjLmNvbHVtbnM6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIHJhdyA9IHNyY1trZXldLnJlaW5kZXgodWlkcykudG9fbnVtcHkoZHR5cGU9bnAuZmxvYXQ2")
parts.append("NCkKICAgICAgICAgICAgICAgIGNsID0gX2NsZWFuKHJhdykKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIGNsID0gX3BpbGtfc29mdChzcmMsIGMpLnJlaW5kZXgodWlkcykudG9fbnVtcHkoZHR5cGU9bnAuZmxvYXQ2NCkKICAgICAgICAgICAgICAg")
parts.append("IGNsID0gX2NsZWFuKGNsKQogICAgICAgICAgICBwYXJ0cy5hcHBlbmQoY2wpCiAgICAgICAgICAgIHdzLmFwcGVuZCh3KQoKICAgICAgICBzdGFjayA9IG5wLnZzdGFjayhwYXJ0cykgICMgKFMsIE4pCiAgICAgICAgd3ZlYyA9IG5wLmFzYXJyYXkod3MsIGR0eXBl")
parts.append("PW5wLmZsb2F0NjQpWzosIE5vbmVdCiAgICAgICAgdmFsaWQgPSBucC5pc2Zpbml0ZShzdGFjaykKICAgICAgICB3c3VtID0gbnAud2hlcmUodmFsaWQsIHd2ZWMsIDAuMCkuc3VtKGF4aXM9MCkKICAgICAgICBwc3VtID0gbnAud2hlcmUodmFsaWQsIHN0YWNrICog")
parts.append("d3ZlYywgMC4wKS5zdW0oYXhpcz0wKQogICAgICAgIGJsZW5kID0gbnAud2hlcmUod3N1bSA+IDAsIHBzdW0gLyBucC5tYXhpbXVtKHdzdW0sIDFlLTkpLCBucC5uYW4pCgogICAgICAgIGRmLmxvY1ttYXNrLCBjXSA9IGJsZW5kCiAgICAgICAgbl91c2UgPSBpbnQo")
parts.append("bnAuaXNmaW5pdGUoYmxlbmQpLnN1bSgpKQogICAgICAgIG5fY2VsbHMgKz0gbl91c2UKICAgICAgICBpZiB2ZXJib3NlOgogICAgICAgICAgICBhZ3JlZSA9IHZhbGlkLmFsbChheGlzPTApLnN1bSgpIGlmIHZhbGlkLnNpemUgZWxzZSAwCiAgICAgICAgICAgIHBy")
parts.append("aW50KAogICAgICAgICAgICAgICAgZiIgIHtjOjE4c30gZ29sZCB7aW50KHRyYWluW2NdLm5vdG5hKCkuc3VtKCkpOjRkfSIKICAgICAgICAgICAgICAgIGYiICsgbXVsdGkge25fdXNlOjRkfSAoYWxsMz17aW50KGFncmVlKX0pID0ge2ludChkZltjXS5ub3RuYSgp")
parts.append("LnN1bSgpKTo1ZH0iCiAgICAgICAgICAgICkKICAgIGlmIHZlcmJvc2U6CiAgICAgICAgcHJpbnQoZiJbRVhQLTAxM10gbXVsdGktc291cmNlIGNlbGxzOiB7bl9jZWxsc30iKQogICAgICAgICMgc2hvdyBsb3NzLXdlaWdodCBtYXNzCiAgICAgICAgeSA9IGRmW0xB")
parts.append("QkVMU10udG9fbnVtcHkoZHR5cGU9bnAuZmxvYXQ2NCkKICAgICAgICB3ID0gMi4wICogbnAuYWJzKHkgLSAwLjUpCiAgICAgICAgcHJpbnQoZiJbRVhQLTAxM10gbG9zcy13IG1lYW49e25wLm5hbm1lYW4odyk6LjNmfSAiCiAgICAgICAgICAgICAgZiJwMTA9e25w")
parts.append("Lm5hbnF1YW50aWxlKHcsMC4xKTouM2Z9IHA5MD17bnAubmFucXVhbnRpbGUodywwLjkpOi4zZn0iKQogICAgcmV0dXJuIGRmCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHJvb3QgPSBQYXRoKG9zLmVudmlyb24uZ2V0KCJSU05BX1JPT1QiLCByIkQ6")
parts.append("L+ahjOmdoi9LYWdnbGUvcnNuYV9rbmVlIikpCiAgICBvdXQgPSBidWlsZF9leHAwMTNfbGFiZWxzKHJvb3QgLyAidHJhaW4uY3N2IikKICAgIGRzdCA9IHJvb3QgLyAibGFiZWxzIiAvICJ0cmFpbl9leHAwMTNfbXVsdGlzb3VyY2UuY3N2IgogICAgZHN0LnBhcmVu")
parts.append("dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBvdXQudG9fY3N2KGRzdCwgaW5kZXg9RmFsc2UpCiAgICBwcmludCgid3JvdGUiLCBkc3QsIG91dC5zaGFwZSkK")
open('/kaggle/working/exp013_multisource.py','wb').write(base64.b64decode(''.join(parts)))
print('wrote exp013_multisource.py', os.path.getsize('/kaggle/working/exp013_multisource.py'))
open('/kaggle/working/rsna_report_labels.py','w').write('# stub\n')
print('embeds ready')


In [ ]:
import os, glob
need = ['llm_labels_v4_blend.csv', 'labels_llm_gpt56sol.csv', 'report_labels_v2.csv']
for fn in need:
    hits = glob.glob(f'/kaggle/input/**/{fn}', recursive=True)
    print(fn, hits)
    assert hits, f'missing {fn}'
os.environ['EXP013_MULTISOURCE'] = '1'
print('EXP013_MULTISOURCE=1')


In [ ]:
import subprocess, sys, os, json
os.chdir('/kaggle/working')
os.environ['EXP013_MULTISOURCE'] = '1'
cmd = [sys.executable, '-u', 'rsna_v3.py',
       '--planes', 'axial',
       '--epochs', '3', '--folds', '5', '--only-fold', '0',
       '--batch', '8', '--n-clips', '4',
       '--experiment-id', 'exp_013_multisource_axial_fold0',
       '--verbose']
print('=== EXP-013 multi-source + Axial fold0 n_clips=4 ===', flush=True)
print(' '.join(cmd), flush=True)
r = subprocess.run(cmd)
print('returncode', r.returncode, flush=True)

import numpy as np, pandas as pd
artifact_root = '/kaggle/working/artifacts/exp_013_multisource_axial_fold0'
for d in [os.path.join(artifact_root, 'fold0')]:
    ok = all(os.path.isfile(os.path.join(d, n)) for n in ['test_pred.npy', 'test_uid.csv', 'meta.json'])
    wok = os.path.isfile(os.path.join(d, 'Axial', 'model.pt'))
    print(f'[artifact] {d}: meta={ok} weights={wok}')
    if ok:
        meta = json.load(open(os.path.join(d, 'meta.json'), encoding='utf-8'))
        pred = np.load(os.path.join(d, 'test_pred.npy'))
        print(' meta', meta)
        print(' pred', pred.shape, float(pred.mean()), bool(np.isfinite(pred).all()))
        assert meta.get('completion_status') == 'done'
if os.path.isfile('/kaggle/working/submission.csv'):
    s = pd.read_csv('/kaggle/working/submission.csv')
    print('[sub]', s.shape)
print('[exp013] done')
